In [ ]:
pip install pyarrow dask[parquet] scikit-learn matplotlib openpyxl

In [ ]:
import os
from pathlib import Path
import re
import ast
from collections.abc import Iterable
import json
import gc
import pandas as pd
import math
import dask.dataframe as dd
from dask.diagnostics import ProgressBar
from datetime import datetime, timedelta, time
from typing import Optional, Union, List, Dict, Any, Tuple, Set
import pytz
import numpy as np
from tqdm import tqdm
from sklearn.cluster import KMeans
from scipy.signal import find_peaks
from sklearn.neighbors import KernelDensity
import matplotlib.pyplot as plt

from dask.distributed import Client
import dask

dask.config.set({
    "distributed.worker.memory.spill": False,
    "dataframe.shuffle.method": "tasks",
    "scheduler": "synchronous",  # prevent P2P shuffle, which requires distributed workers and crashes with synchronous scheduler
})

client = Client(processes=True)

# Silence the fillna FutureWarning in the main process and in all workers.
# Setting the option is cleaner than a filter because it also silences
# Dask's own internal fillna calls which run inside worker processes.
pd.set_option('future.no_silent_downcasting', True)
client.run(lambda: pd.set_option('future.no_silent_downcasting', True))

# Define the data directory
base_path = Path("/home/odedshah.post.ac.il/Desktop/local_share/lynx-workspace")
data_dir = base_path / "unified_data"

In [ ]:
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pyarrow.compute as pc

TABLE_PATHS = {
    "clinical_events": data_dir / "clinical_events",
    "measurement": data_dir / "measurement_norm",
    "drug_exposure": data_dir / "drug_exposure",
    "visits": data_dir / "visits",
    "visit_types": data_dir / "visit_types",
}

def apply_pattern_filter(df: pd.DataFrame, col: str, pattern):
    """
    pattern can be:
      - string (simple contains)
      - string with % separators -> AND condition
      - list of strings -> OR across items
    """
    series = df[col].astype(str).str.lower()

    def match_and(pattern_str: str):
        parts = [p.strip().lower() for p in pattern_str.split("%") if p.strip()]
        mask = pd.Series(True, index=df.index)
        for part in parts:
            mask &= series.str.contains(part, na=False)
        return mask

    if isinstance(pattern, str):
        if "%" in pattern:
            return df[match_and(pattern)]
        else:
            return df[series.str.contains(pattern.lower(), na=False)]

    if isinstance(pattern, (list, tuple)):
        combined_mask = pd.Series(False, index=df.index)
        for p in pattern:
            if "%" in p:
                combined_mask |= match_and(p)
            else:
                combined_mask |= series.str.contains(p.lower(), na=False)
        return df[combined_mask]

    raise ValueError("Invalid pattern type. Must be str or list of str.")


def _build_pq_filters(filters):
    """Convert filters dict to pyarrow DNF filter tuples (dask's as_dask path)."""
    if not filters:
        return None
    result = []
    for col, val in filters.items():
        if isinstance(val, (list, tuple, set)):
            result.append((col, "in", list(val)))
        else:
            result.append((col, "==", val))
    return result


def _build_pa_expression(filters):
    """
    Convert filters dict to a pyarrow.dataset filter expression.

    A single expression lets pyarrow.dataset compose both pruning mechanisms
    at once: for a Hive-partitioned column (e.g. hospital=X/ folders) it skips
    whole subfolders without opening any file; for any other column it falls
    back to row-group statistics pruning within the files that remain.
    """
    if not filters:
        return None
    expr = None
    for col, val in filters.items():
        cond = pc.field(col).isin(list(val)) if isinstance(val, (list, tuple, set)) else pc.field(col) == val
        expr = cond if expr is None else (expr & cond)
    return expr


def get_data(
    table: str,
    *,
    filters=None,
    name_pattern=None,
    pattern_col=None,
    columns=None,
    top_n=1000,
    as_dask=False,
):
    """
    Dynamic raw query over unified tables.

    Parameters
    ----------
    table : str
        One of {"clinical_events", "measurement", "drug_exposure", "visits", "visit_types"}.
    filters : dict, optional
        Equality / IN filters. On the pandas path these are pushed down via a
        single pyarrow.dataset expression that prunes both Hive partitions
        (e.g. whole hospital=X/ folders) and row-group statistics within the
        remaining files -- both filters compose, no extra work needed by the
        caller.
    name_pattern : str or list[str], optional
        Substring filter applied to pattern_col (pandas-side -- substring
        matches can never be pushed down to parquet stats, only filters does).
    pattern_col : str, optional
        Column to apply name_pattern on.
    columns : list[str], optional
        Subset of columns to load; None = all.
    top_n : int, optional
        Max rows to return (default 1000). None = all matching rows.
    as_dask : bool
        If True -> return lazy Dask DataFrame (no compute).
        If False -> return pandas DataFrame (fast, pyarrow.dataset-driven).
    """
    if table not in TABLE_PATHS:
        raise ValueError(f"Unknown table '{table}', valid: {list(TABLE_PATHS.keys())}")

    path = TABLE_PATHS[table]

    if as_dask:
        pq_filters = _build_pq_filters(filters)
        ddf = dd.read_parquet(path, columns=columns, filters=pq_filters)
        if name_pattern is not None:
            if pattern_col is None:
                raise ValueError("pattern_col must be provided when using name_pattern")
            ddf = ddf.map_partitions(apply_pattern_filter, pattern_col, name_pattern)
        return ddf

    # Pandas path: pyarrow.dataset with Hive partition discovery.
    # scanner() skips whole hospital=X/ folders that can't match the filter,
    # prunes row groups within surviving files via statistics, and correctly
    # reconstructs partition columns (e.g. hospital) from the folder name --
    # fragment.to_table() does NOT do this, which is why we use to_reader().
    # Generalizes to any filter: partition columns get folder-level pruning,
    # physical columns get row-group stats pruning, both compose automatically.
    dataset = ds.dataset(str(path), format="parquet", partitioning="hive")
    pa_expr = _build_pa_expression(filters)
    reader = dataset.scanner(filter=pa_expr, columns=columns).to_reader()

    chunks = []
    collected = 0
    for batch in reader:
        df = batch.to_pandas()

        if name_pattern is not None:
            df = apply_pattern_filter(df, pattern_col, name_pattern)

        if len(df):
            chunks.append(df)
            collected += len(df)
            if top_n is not None and collected >= top_n:
                break
        del df

    result = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
    return result.head(top_n) if top_n is not None else result

In [ ]:
pdf = get_data(
    "measurement",
    filters={"hospital": ""},
    name_pattern="glucose",
    pattern_col="concept_name",
    columns=["person_id", "visit_id", "concept_id", "concept_name", "unit", "value", "start_datetime", "hospital"],
    top_n=10,
    as_dask=False,
)
pdf

# Building the Cohort

In [ ]:
def build_visit_master(
    hospital: Optional[str] = None,
    min_duration_hours: int = 48,
    max_duration_hours: int = 14 * 24,
    save_csv: Optional[str] = "visits_master.csv",
    as_dask: bool = False,
):
    """
    Annotate visits with:
      - duration_days (float)
      - relevant_department (bool)   -> False if visit ever in an excluded dept
      - relevant_duration (bool)    -> within [min_duration_hours, max_duration_hours]
      - is_inpatient (bool)

    No visits are dropped.
    """

    # 1. Load visits
    filters = {"hospital": hospital} if hospital is not None else None

    visits_ddf = get_data(
        table="visits",
        filters=filters,
        columns=[
            "person_id",
            "visit_id",
            "hospital",
            "visit_type",
            "start_datetime",
            "end_datetime",
            "birth_datetime",
            "gender",
            "death_datetime"
        ],
        as_dask=True,
    )

    # 2. Duration in days
    print("=== CALCULATING VISIT DURATION ===")
    visits_ddf["start_datetime"] = dd.to_datetime(visits_ddf["start_datetime"], utc=True)
    visits_ddf["end_datetime"] = dd.to_datetime(visits_ddf["end_datetime"], utc=True)

    duration_hours = (
        visits_ddf["end_datetime"] - visits_ddf["start_datetime"]
    ) / np.timedelta64(1, "h")

    visits_ddf["duration_days"] = duration_hours / 24.0

    # 2b. Death within 30 days after discharge
    print("=== FLAGGING DEATH WITHIN 30 DAYS OF DISCHARGE ===")
    visits_ddf["death_datetime"] = dd.to_datetime(
        visits_ddf["death_datetime"], utc=True
    )

    delta_death_days = (
        visits_ddf["death_datetime"] - visits_ddf["end_datetime"]
    ) / np.timedelta64(1, "D")

    delta_death_hours = (
        visits_ddf["death_datetime"] - visits_ddf["start_datetime"]
    ) / np.timedelta64(1, "h")

    visits_ddf["death_within_30d"] = (delta_death_days >= -14) & (delta_death_days <= 30)
    visits_ddf["death_within_48h"] = (delta_death_hours >= 0) & (delta_death_hours <= 48)

    # 3. Relevant department flag
    print("=== FLAGGING RELEVANT DEPARTMENTS ===")
    visit_types_ddf = get_data(
        table="visit_types",
        filters=filters,
        columns=["visit_id", "care_site_name"],
        as_dask=True,
    )

    irrelevant_substrings = ["", "", ""]
    pattern = "|".join(irrelevant_substrings)

    mask = visit_types_ddf["care_site_name"].fillna("").str.contains(pattern, regex=True)
    excluded_ids_ddf = (
        visit_types_ddf[mask][["visit_id"]]
        .drop_duplicates()
        .assign(relevant_department=False)
    )

    # Compute excluded visit_ids to pandas for a broadcast merge -- avoids a
    # Dask-to-Dask merge which triggers a distributed P2P shuffle that crashes.
    excluded_ids_pd = excluded_ids_ddf.compute(scheduler="synchronous")

    def _flag_excluded(pdf):
        merged = pdf.merge(excluded_ids_pd, on="visit_id", how="left")
        merged["relevant_department"] = merged["relevant_department"].fillna(True).astype(bool)
        return merged

    visits_ddf = visits_ddf.map_partitions(_flag_excluded)

    # 4. Relevant duration flag
    visits_ddf["relevant_duration"] = (
        (duration_hours >= min_duration_hours)
        & (duration_hours <= max_duration_hours)
    )

    # 5. Inpatient flag
    visits_ddf["is_inpatient"] = visits_ddf["visit_type"] == "Inpatient Visit"

    # 6. Adult at admission (>=18 years)
    visits_ddf["birth_datetime"] = dd.to_datetime(visits_ddf["birth_datetime"], utc=True)
    age_days = (
        visits_ddf["start_datetime"] - visits_ddf["birth_datetime"]
    ) / np.timedelta64(1, "D")
    visits_ddf["adult_at_admission"] = age_days >= (18 * 365.25)

    if as_dask:
        return visits_ddf

    df = visits_ddf.compute(scheduler="synchronous")

    if save_csv is not None:
        df.to_csv(save_csv, index=False)

    return df

In [ ]:
def load_concept_ids_from_file(path):
    """Return list of int concept_ids from a cohort_filters file."""
    with open(path, encoding="utf-8") as f:
        text = f.read()
    ids = re.findall(r"ID:\s*(\d+)", text)
    return [str(x) for x in ids]

def _compute_glucose_flags_per_visit(group, high, hyper, low, hypo):
    times = group["delta_hours"].values
    vals = pd.to_numeric(group["value"], errors="coerce").values
    vals = vals[~np.isnan(vals)]
    times = times[: len(vals)]

    if len(vals) == 0:
        return pd.Series(
            {"has_extreme_glucose": False, "has_repeated_high_low": False}
        )

    has_extreme = ((vals >= hyper) | (vals <= hypo)).any()

    def has_pair(mask):
        idx = np.where(mask)[0]
        if len(idx) < 2:
            return False
        t = np.sort(times[idx])
        for i in range(len(t)):
            for j in range(i + 1, len(t)):
                dt = t[j] - t[i]
                if dt < 2:
                    continue
                if dt > 24:
                    break
                return True
        return False

    high_pair = has_pair(vals >= high)
    low_pair = has_pair(vals <= low)
    has_repeated = high_pair or low_pair

    return pd.Series(
        {"has_extreme_glucose": bool(has_extreme),
         "has_repeated_high_low": bool(has_repeated)}
    )

def add_glucose_and_diabetes_flags(
    visits_ddf,
    hospital=None,
    high_glucose_val=180,
    hyper_glucose_val=250,
    low_glucose_val=0, # 70 -> muted, cohort now focuses only on hyper as entry criteria
    hypo_glucose_val=0, # 54
    glucose_ids_file="cohort_filters/include_GLUCOSE_TEST.txt",
    diabetes_ids_file="cohort_filters/include_DD.txt",
):
    """
    Adds to visits_ddf (Dask):

      - has_extreme_glucose
      - has_repeated_high_low
      - has_diabetes_dx
    """

    filters = {"hospital": hospital} if hospital is not None else None

    # Load visit times ONCE as pandas for all broadcast merges below.
    # Using get_data (pyarrow.dataset path) avoids embedding visits_ddf's
    # lazy graph into measurement merge graphs, which would force full
    # materialization of the enriched visits on every compute() call.
    visit_times_pd = get_data(
        "visits",
        filters=filters,
        columns=["visit_id", "start_datetime", "end_datetime"],
        top_n=None,
    )

    # ---- Glucose from measurements ----
    print("=== LOADING DATA ===")
    glucose_ids = load_concept_ids_from_file(glucose_ids_file)
    print(f"=== MATCHING WITH {len(glucose_ids)} IDs ===")

    meas_ddf = get_data(
        table="measurement",
        filters=filters,
        columns=[
            "visit_id",
            "concept_id",
            "concept_name",
            "start_datetime",
            "value",
        ],
        as_dask=True,
    )

    meas_ddf = meas_ddf[meas_ddf["concept_id"].isin(glucose_ids)]
    meas_ddf = meas_ddf.rename(columns={"start_datetime": "measurement_datetime"})

    # Broadcast merge: pandas right side, no P2P shuffle triggered.
    visit_start_pd = visit_times_pd[["visit_id", "start_datetime"]].rename(
        columns={"start_datetime": "visit_start_datetime"}
    )
    meas_ddf = meas_ddf.merge(visit_start_pd, on="visit_id", how="inner")

    meas_ddf["measurement_datetime"] = dd.to_datetime(meas_ddf["measurement_datetime"], utc=True)
    meas_ddf["visit_start_datetime"] = dd.to_datetime(meas_ddf["visit_start_datetime"], utc=True)

    delta_hours = (
        meas_ddf["measurement_datetime"] - meas_ddf["visit_start_datetime"]
    ) / np.timedelta64(1, "h")

    meas_ddf = meas_ddf[(delta_hours >= 0) & (delta_hours <= 48)]
    meas_ddf = meas_ddf.assign(delta_hours=delta_hours)

    print("=== CALCULATING GLUCOSE CONDITION ===")
    # Synchronous scheduler: one partition at a time, avoids distributed P2P.
    meas_df = meas_ddf.compute(scheduler="synchronous")

    if len(meas_df) == 0:
        visits_ddf = visits_ddf.assign(
            has_extreme_glucose=False, has_repeated_high_low=False
        )
    else:
        glucose_flags = (
            meas_df.groupby("visit_id")
            .apply(
                _compute_glucose_flags_per_visit,
                high=high_glucose_val,
                hyper=hyper_glucose_val,
                low=low_glucose_val,
                hypo=hypo_glucose_val,
                include_groups=False,
            )
            .reset_index()
        )

        # glucose_flags is pandas -- Dask broadcasts it automatically, no P2P.
        visits_ddf = visits_ddf.merge(glucose_flags, on="visit_id", how="left")
        visits_ddf["has_extreme_glucose"] = visits_ddf["has_extreme_glucose"].fillna(False)
        visits_ddf["has_repeated_high_low"] = visits_ddf["has_repeated_high_low"].fillna(False)

    # ---- Diabetes diagnoses from clinical_events ----
    print("=== LOADING CLINICAL EVENTS ===")
    diabetes_ids = load_concept_ids_from_file(diabetes_ids_file)
    print(f"=== MATCHING WITH {len(diabetes_ids)} IDs ===")

    ce_ddf = get_data(
        table="clinical_events",
        filters=filters,
        columns=["visit_id", "concept_id", "concept_name", "start_datetime"],
        as_dask=True,
    )
    print("=== FINDING RELEVANT DIAGNOSIS ===")
    ce_ddf = ce_ddf[ce_ddf["concept_id"].isin(diabetes_ids)]

    diabetes_visits = (
        ce_ddf[["visit_id"]]
        .drop_duplicates()
        .assign(has_diabetes_dx=True)
    )

    # Compute to pandas for broadcast merge -- avoids Dask-to-Dask P2P shuffle.
    diabetes_visits_pd = diabetes_visits.compute(scheduler="synchronous")
    visits_ddf = visits_ddf.merge(diabetes_visits_pd, on="visit_id", how="left")
    visits_ddf["has_diabetes_dx"] = visits_ddf["has_diabetes_dx"].fillna(False)

    # ---- Night measurement check ----
    print("=== ENSURING MEASUREMENTS WERE TAKEN OVERNIGHT ===")
    meas_any_ddf = get_data(
        table="measurement",
        filters=filters,
        columns=["visit_id", "start_datetime"],
        as_dask=True,
    ).rename(columns={"start_datetime": "measurement_datetime"})

    # Broadcast merge with pandas visit window -- no P2P.
    visit_window_pd = visit_times_pd[["visit_id", "start_datetime", "end_datetime"]].rename(
        columns={
            "start_datetime": "visit_start_datetime",
            "end_datetime": "visit_end_datetime",
        }
    )
    meas_any_ddf = meas_any_ddf.merge(visit_window_pd, on="visit_id", how="inner")

    meas_any_ddf["measurement_datetime"] = dd.to_datetime(
        meas_any_ddf["measurement_datetime"]
    ).dt.tz_localize(None)
    meas_any_ddf["visit_start_datetime"] = dd.to_datetime(
        meas_any_ddf["visit_start_datetime"]
    ).dt.tz_localize(None)
    meas_any_ddf["visit_end_datetime"] = dd.to_datetime(
        meas_any_ddf["visit_end_datetime"]
    ).dt.tz_localize(None)

    inside_visit = (
        (meas_any_ddf["measurement_datetime"] >= meas_any_ddf["visit_start_datetime"])
        & (meas_any_ddf["measurement_datetime"] <= meas_any_ddf["visit_end_datetime"])
    )
    meas_any_ddf = meas_any_ddf[inside_visit]

    hour = meas_any_ddf["measurement_datetime"].dt.hour
    night_mask = (hour >= 18) | (hour < 7)

    night_visits_ddf = (
        meas_any_ddf[night_mask][["visit_id"]]
        .drop_duplicates()
        .assign(has_night_measure=True)
    )

    # Compute to pandas for broadcast merge -- no P2P.
    night_visits_pd = night_visits_ddf.compute(scheduler="synchronous")
    visits_ddf = visits_ddf.merge(night_visits_pd, on="visit_id", how="left")
    visits_ddf["has_night_measure"] = visits_ddf["has_night_measure"].fillna(False)

    visits_ddf["is_inpatient"] = visits_ddf["is_inpatient"] & visits_ddf["has_night_measure"]

    visits_ddf["diabetic_condition"] = (
        visits_ddf["has_extreme_glucose"]
        | visits_ddf["has_repeated_high_low"]
        | visits_ddf["has_diabetes_dx"]
    )
    return visits_ddf

In [ ]:
# 1. Build visits master (annotated, Dask)
visits_ddf = build_visit_master(hospital=None, save_csv=None, as_dask=True)

# 2. Add glucose + diabetes flags
visits_ddf = add_glucose_and_diabetes_flags(
    visits_ddf,
    hospital=None,
)

# 3. Materialize
visits_df = visits_ddf.compute()

In [ ]:
# A) Broken in visits_master: visit_id maps to multiple person_id
vm_map = (
    visits_df.groupby("visit_id")["person_id"]
    .nunique(dropna=True)
    .rename("n_persons_per_visit")
    .reset_index()
)
vm_map["broken_visit_id_vm"] = vm_map["n_persons_per_visit"] > 1

# Merge flags into visits_master
visits_df = visits_df.merge(vm_map, on="visit_id", how="left")

# Final broken flag (broken in either source)
visits_df["broken_visit_id"] = (
    visits_df["broken_visit_id_vm"].fillna(False)
).fillna(False)

print("Broken visit_id counts:")
print(visits_df["broken_visit_id"].value_counts(dropna=False))

remove_columns = ["broken_visit_id_vm", "n_persons_per_visit"]
visits_df = visits_df.drop(columns=[col for col in remove_columns if col in visits_df.columns])

visits_df["relevant_admission"] = (
    visits_df["is_inpatient"]
    & ~visits_df["broken_visit_id"]
    & visits_df["relevant_department"]
    & visits_df["relevant_duration"]
    & ~visits_df["death_within_48h"]
    & visits_df["adult_at_admission"]
    & visits_df["diabetic_condition"]
)
visits_df.to_csv("visits_master.csv", index=False)

# Query ICU visits
cohort = visits_df[
    visits_df["is_inpatient"]
    & ~visits_df["broken_visit_id"]    
    & visits_df["relevant_department"]
    & visits_df["relevant_duration"]
    & ~visits_df["death_within_48h"]
    & visits_df["adult_at_admission"]
    & visits_df["diabetic_condition"]
]
print("<==========================================================>")
print("Cohort Patient ID by Hospital, ", "N=", cohort["person_id"].nunique())
print(cohort.groupby("hospital")["person_id"].nunique())
print("")
print("Cohort admission ID by Hospital, ", "N=", cohort["visit_id"].nunique())
print(cohort.groupby("hospital")["visit_id"].nunique())

In [ ]:
# Query ICU visits

visits_df = pd.read_csv("visits_master.csv", low_memory=False)

# Build mask on the full DF
mask = (
    visits_df["is_inpatient"]
    & ~visits_df["broken_visit_id"]
    & visits_df["relevant_department"]
    & visits_df["relevant_duration"]
    & ~visits_df["death_within_48h"]
    & visits_df["adult_at_admission"]
    & visits_df["diabetic_condition"]
)

# Take a real copy to avoid SettingWithCopyWarning
admissions = visits_df.loc[mask].copy()

# Convert start date to timezone naive datetime
admissions["start_datetime"] = (
    pd.to_datetime(admissions["start_datetime"], utc=True)
      .dt.tz_convert(None)
)

# Filter for 2020 and up
admissions = admissions[admissions["start_datetime"].dt.year >= 2020]

# Month year column (no timezone now, so no warning)
admissions["month_year"] = admissions["start_datetime"].dt.to_period("M")

# Count visits per month
monthly_counts = admissions["month_year"].value_counts().sort_index()

# Plot
monthly_counts.plot(kind="bar", figsize=(12, 5))
plt.title(f"Patient Visit Distribution by Month Year (N={len(admissions)})")
plt.xlabel("Month Year")
plt.ylabel("Number of Visits")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Converting from DB to Mediator Input

## Building Temporal Dataset

In [ ]:
def parse_line(line):
    """Return the ID part as string from a mapping line, or None."""
    for part in line.split("|"):
        part = part.strip()
        if part.lower().startswith("id:"):
            return part[3:].strip()
    return None

def _load_file_dicts(file_path):
    """
    Load one mapping file that may contain:
      - a single dict
      - a list of dicts
      - multiple top-level dicts one after another:  {..}{..}{..}

    Returns a list of dicts.
    On any parsing problem, prints a detailed warning and skips the broken piece.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    dict_strings = []
    depth = 0
    start_idx = None

    # 1) First, try to slice out all top-level {...} blocks by brace depth
    for i, ch in enumerate(content):
        if ch == "{":
            if depth == 0:
                start_idx = i
            depth += 1
        elif ch == "}":
            if depth > 0:
                depth -= 1
                if depth == 0 and start_idx is not None:
                    dict_strings.append(content[start_idx : i + 1])
                    start_idx = None

    dicts = []

    if dict_strings:
        # We found one or more dict blocks; parse each separately
        for idx, dstr in enumerate(dict_strings):
            try:
                data = ast.literal_eval(dstr)
                if isinstance(data, dict):
                    dicts.append(data)
                else:
                    print(
                        f"[WARNING] In '{file_path}', block #{idx} is not a dict after parsing. "
                        f"Type: {type(data).__name__}. Skipping."
                    )
            except Exception as e:
                print(
                    f"[WARNING] Could not parse dict block #{idx} in '{file_path}' "
                    f"as Python literal. Error: {type(e).__name__}: {e}"
                )
                # Show a short snippet of the problematic block
                lines = dstr.splitlines()
                n = len(lines)
                print("         Block snippet:")
                for j in range(min(n, 8)):
                    print(f"           {j+1:03d}: {lines[j]}")
                print("---------")
        return dicts

    # 2) Fallback: no top-level dicts detected (unusual). Try classic literal_eval.
    try:
        data = ast.literal_eval(content)
    except Exception as e:
        print(f"[WARNING] Could not parse mapping content in '{file_path}' as Python literal.")
        print(f"         Error: {type(e).__name__}: {e}")
        lines = content.splitlines()
        n = len(lines)
        print("         File snippet:")
        for i in range(min(n, 8)):
            print(f"           {i+1:03d}: {lines[i]}")
        print("---------")
        return []

    if isinstance(data, dict):
        return [data]
    if isinstance(data, list):
        return [d for d in data if isinstance(d, dict)]

    print(f"[WARNING] Parsed mapping in '{file_path}' is neither dict nor list. Ignoring.")
    return []

def load_index_specs(input_path: str) -> Dict[str, Dict[str, List[Dict[str, Any]]]]:
    """
    Load all index definition files into a nested dict:

        specs[table_name][mediator_concept_name] = list of rule_dicts

    Each rule_dict:
        {
            "conditions": [
                {"column": <str>, "op": "in_ids", "ids": [<str>, ...]},
                {"column": <str>, "op": "eq",     "values": [<scalar>, ...]},
                ...
            ],
            "source_file": <file_name>,
            "rule_index": <int>
        }

    Notes:
      * Folder structure: input_path/<table_name>/*.txt
      * Each .txt is one or more Python dict literals.
      * Dict keys are column names; values can be:
          - list of 'ID: ... | Value: ...' strings (concept lists)
          - simple scalars (equality filters)
      * Keys 'ignore_values' and 'bins' are ignored.
    """
    specs: Dict[str, Dict[str, List[Dict[str, Any]]]] = {}
    total_ids = 0
    total_concepts = 0

    for table_name in os.listdir(input_path):
        table_path = os.path.join(input_path, table_name)
        if not os.path.isdir(table_path):
            continue

        for file_name in os.listdir(table_path):
            if not file_name.lower().endswith(".txt"):
                continue

            mediator_concept = os.path.splitext(file_name)[0]
            file_path = os.path.join(table_path, file_name)

            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()

            dicts_in_file = _load_file_dicts(file_path)
            if not dicts_in_file:
                continue

            for rule_idx, raw_dict in enumerate(dicts_in_file):
                # Drop metadata keys we do not need
                raw_dict = {
                    k: v
                    for k, v in raw_dict.items()
                    if k not in ("ignore_values", "bins")
                }

                conditions: List[Dict[str, Any]] = []

                for column_name, value in raw_dict.items():

                    # Case 1: iterable value (list / tuple / set), not a plain string
                    if isinstance(value, Iterable) and not isinstance(value, (str, bytes)):
                        seq = list(value)
                        if not seq:
                            continue

                        first = seq[0]

                        # 1a. Iterable of ID lines ("ID: ... | Value: ...")
                        if isinstance(first, str) and first.lstrip().lower().startswith("id:"):
                            ids: List[str] = []
                            for line in seq:
                                if not isinstance(line, str):
                                    continue
                                db_id = parse_line(line)
                                if db_id:
                                    ids.append(db_id)

                            if ids:
                                total_ids += len(ids)
                                # For ID lists we ALWAYS filter on concept_id, regardless of the mapping key name
                                conditions.append(
                                    {
                                        "column": "concept_id",
                                        "op": "in_ids",
                                        "ids": ids,      # IDs kept as *strings*
                                    }
                                )

                        # 1b. Iterable of scalar filter values (e.g. route, drug_type_category)
                        else:
                            conditions.append(
                                {
                                    "column": column_name,
                                    "op": "eq",
                                    "values": seq,
                                }
                            )

                    # Case 2: scalar value (string, int, etc.)
                    else:
                        conditions.append(
                            {
                                "column": column_name,
                                "op": "eq",
                                "values": [value],
                            }
                        )

                if not conditions:
                    continue

                rule_dict = {
                    "conditions": conditions,
                    "source_file": file_name,
                    "rule_index": rule_idx,
                }

                specs.setdefault(table_name, {}).setdefault(mediator_concept, []).append(
                    rule_dict
                )
                total_concepts += 1

    print(
        f"Loaded index specs from '{input_path}': "
        f"{total_concepts} concept-rule entries with {total_ids} mapped IDs (with duplicates)."
    )

    return specs

In [ ]:
def filter_patient_events(events_df: pd.DataFrame,
                          concept_col: str = "ConceptName") -> pd.DataFrame:
    """
    For each (PatientId, VisitId), keep only the range between the first
    ADMISSION and the last RELEASE/DEATH token, inclusive.

    events_df must have:
      - PatientId
      - VisitId
      - ConceptName (or override via concept_col)
      - StartDateTime

    Returns a filtered DataFrame with the same columns.
    """
    events_df = events_df.copy()

    # Ensure sortable dtypes
    events_df["PatientId"] = events_df["PatientId"].astype("int64")
    events_df["VisitId"] = events_df["VisitId"].astype("int64")
    # Ensure datetime
    events_df["StartDateTime"] = pd.to_datetime(
        events_df["StartDateTime"], errors="coerce", utc=True
    )

    events_df = events_df.sort_values(
        by=["PatientId", "VisitId", "StartDateTime"]
    ).reset_index(drop=True)

    def _clip_one(group: pd.DataFrame) -> pd.DataFrame:
        tokens = group[concept_col].tolist()
        try:
            start_idx = tokens.index("ADMISSION")
        except ValueError:
            # no ADMISSION token ? drop this visit
            return pd.DataFrame(columns=group.columns)

        # last RELEASE or DEATH
        end_idx = None
        for i in range(len(tokens) - 1, -1, -1):
            if tokens[i] in {"RELEASE", "DEATH"}:
                end_idx = i
                break

        if end_idx is None or end_idx <= start_idx:
            return pd.DataFrame(columns=group.columns)

        return group.iloc[start_idx : end_idx + 1]

    filtered_groups = []
    for _, group in events_df.groupby(["PatientId", "VisitId"], sort=False):
        clipped = _clip_one(group)
        if not clipped.empty:
            filtered_groups.append(clipped)

    if not filtered_groups:
        return pd.DataFrame(columns=events_df.columns)
    _result = pd.concat(filtered_groups, ignore_index=True)
    _result["PatientId"] = _result["PatientId"].astype(str)
    _result["VisitId"]   = _result["VisitId"].astype(str)
    return _result

def add_fake_meal_rows(visits_df: pd.DataFrame,
                       id_column: str = "person_id",
                       visit_column: str = "visit_id") -> pd.DataFrame:
    """
    For each visit in visits_df, generate MEAL rows at fixed times
    (07:00, 12:00, 18:00, 22:00) for each day between start_datetime and
    end_datetime, if the meal time falls inside the admission window.

    Returns a DataFrame with columns:
      PatientId, VisitId, ConceptName, StartDateTime, EndDateTime, Value
    """

    time_to_meal = {
        "07:00": "Breakfast",
        "12:00": "Lunch",
        "18:00": "Dinner",
        "22:00": "Night-Snack",
    }
    meal_times = list(time_to_meal.keys())

    def _meals_for_visit(row):
        start = pd.to_datetime(row["start_datetime"], utc=True)
        end   = pd.to_datetime(row["end_datetime"], utc=True)

        current_day = start.normalize()
        last_day    = end.normalize()

        out = []
        while current_day <= last_day:
            for t in meal_times:
                hh, mm = map(int, t.split(":"))
                meal_dt = current_day + pd.Timedelta(hours=hh, minutes=mm)

                if start <= meal_dt <= end:
                    out.append({
                        "PatientId":   row[id_column],
                        "VisitId":     row[visit_column],
                        "ConceptName": "MEAL",
                        "StartDateTime": meal_dt,
                        "EndDateTime":   meal_dt + pd.Timedelta(seconds=1),
                        "Value":       time_to_meal[t],
                    })
            current_day += pd.Timedelta(days=1)

        return out

    rows_series = visits_df.apply(_meals_for_visit, axis=1)
    meal_rows = [d for sublist in rows_series for d in sublist]

    if not meal_rows:
        return pd.DataFrame(columns=[
            "PatientId", "VisitId", "ConceptName",
            "StartDateTime", "EndDateTime", "Value"
        ])

    return pd.DataFrame(meal_rows)

def build_admission_release_rows(visits_df: pd.DataFrame,
                                 id_column: str = "person_id",
                                 visit_column: str = "visit_id") -> pd.DataFrame:
    """
    For each row in visits_df, create:
      - ADMISSION token at start_datetime
      - DEATH or RELEASE token at end_datetime (+1 sec), depending on
        death_within_30d flag.

    Returns a DataFrame with:
      PatientId, VisitId, ConceptName, StartDateTime, EndDateTime, Value
    """

    def _rows_for_visit(row):
        start = pd.to_datetime(row["start_datetime"], utc=True)
        end   = pd.to_datetime(row["end_datetime"], utc=True)

        out = []

        # ADMISSION
        out.append({
            "PatientId":   row[id_column],
            "VisitId":     row[visit_column],
            "ConceptName": "ADMISSION",
            "StartDateTime": start,
            "EndDateTime":   start + pd.Timedelta(seconds=1),
            "Value": True,
        })

        # RELEASE or DEATH
        is_death = bool(row.get("death_within_30d", False))
        end_token = "DEATH" if is_death else "RELEASE"

        out.append({
            "PatientId":   row[id_column],
            "VisitId":     row[visit_column],
            "ConceptName": end_token,
            "StartDateTime": end,
            "EndDateTime":   end + pd.Timedelta(seconds=1),
            "Value": True,
        })

        return out

    rows_series = visits_df.apply(_rows_for_visit, axis=1)
    rows = [d for sublist in rows_series for d in sublist]

    return pd.DataFrame(rows, columns=[
        "PatientId", "VisitId", "ConceptName",
        "StartDateTime", "EndDateTime", "Value"
    ])

def propagate_prior_concepts(
    events_df: pd.DataFrame,
    visits_df: pd.DataFrame,
    concept_windows: dict,
    id_col_events: str = "PatientId",
    visit_col_events: str = "VisitId",
    id_col_visits: str = "person_id",
    visit_col_visits: str = "visit_id",
    start_col_visits: str = "start_datetime",
    target_only_relevant: bool = True,
) -> pd.DataFrame:
    """
    For each (patient, visit, concept) in concept_windows:
      - If that concept is NOT present in the current admission,
      - Look back in *previous* admissions of the same patient,
        within the given window (in days),
      - If any events exist, take ONLY the last one (latest StartDateTime),
      - Inject a single synthetic row into the current admission with:
            StartDateTime = admission_start + 1s
            EndDateTime   = admission_start + 2s
            Value         = last historical Value

    Returns a DataFrame of propagated rows in the same schema as events_df.
    """

    visits = visits_df.copy()
    visits[start_col_visits] = pd.to_datetime(visits[start_col_visits], utc=True)

    events_df = events_df.copy()
    events_df["StartDateTime"] = pd.to_datetime(events_df["StartDateTime"], utc=True)

    ev_by_patient = dict(tuple(events_df.groupby(id_col_events)))
    visits_by_patient = dict(tuple(visits.groupby(id_col_visits)))

    rows = []

    for pid, v_grp in visits_by_patient.items():
        if pid not in ev_by_patient:
            continue

        patient_events = ev_by_patient[pid].sort_values("StartDateTime")
        v_grp = v_grp.sort_values(start_col_visits)

        events_by_visit = {
            vid: g["ConceptName"].unique().tolist()
            for vid, g in patient_events.groupby(visit_col_events)
        }

        for _, v_row in v_grp.iterrows():
            vid = v_row[visit_col_visits]
            adm_start = v_row[start_col_visits]

            # Only inject into relevant admissions if requested
            if target_only_relevant and not bool(v_row.get("relevant_admission", True)):
                continue

            existing_concepts = set(events_by_visit.get(vid, []))

            for concept, window_days in concept_windows.items():
                if concept in existing_concepts:
                    continue

                lower_bound = adm_start - pd.Timedelta(days=window_days)

                hist = patient_events[
                    (patient_events["ConceptName"] == concept)
                    & (patient_events["StartDateTime"] < adm_start)
                    & (patient_events["StartDateTime"] >= lower_bound)
                ]
                if hist.empty:
                    continue

                last_ev = hist.sort_values("StartDateTime").iloc[-1]

                rows.append(
                    {
                        "PatientId":     pid,
                        "VisitId":       v_row[visit_col_visits],
                        "ConceptName":   concept,
                        "StartDateTime": adm_start + pd.Timedelta(seconds=1),
                        "EndDateTime":   adm_start + pd.Timedelta(seconds=2),
                        "Value":         last_ev["Value"],
                    }
                )

    if not rows:
        return pd.DataFrame(columns=events_df.columns)

    return pd.DataFrame(rows, columns=events_df.columns)

def add_bmi_rows_from_events(events_df: pd.DataFrame,
                             weight_concept: str = "WEIGHT_MEASURE",
                             height_concept: str = "HEIGHT_MEASURE",
                             bmi_concept: str = "BMI_MEASURE") -> pd.DataFrame:
    """
    Compute BMI rows from temporal WEIGHT and HEIGHT events.

    Enhancement:
      - If BMI_MEASURE already exists in (PatientId, VisitId), skip that admission.
    """

    df = events_df.copy()
    df["StartDateTime"] = pd.to_datetime(df["StartDateTime"], utc=True)
    df["EndDateTime"]   = pd.to_datetime(df["EndDateTime"], utc=True)

    # 1. Identify admission-level BMI presence
    bmi_exists = (
        df[df["ConceptName"] == bmi_concept]
        .groupby(["PatientId", "VisitId"])
        .size()
        .reset_index()[["PatientId", "VisitId"]]
    )
    bmi_exists["skip"] = True

    # 2. Filter weight + height rows
    w = df[df["ConceptName"] == weight_concept].copy()
    h = df[df["ConceptName"] == height_concept].copy()

    if w.empty or h.empty:
        return pd.DataFrame(columns=[
            "PatientId", "VisitId", "ConceptName",
            "StartDateTime", "EndDateTime", "Value"
        ])

    # Remove admissions where BMI already exists
    w = w.merge(bmi_exists, on=["PatientId", "VisitId"], how="left")
    h = h.merge(bmi_exists, on=["PatientId", "VisitId"], how="left")
    w = w[w["skip"].isna()]
    h = h[h["skip"].isna()]

    if w.empty or h.empty:
        return pd.DataFrame(columns=[
            "PatientId", "VisitId", "ConceptName",
            "StartDateTime", "EndDateTime", "Value"
        ])

    # 3. Continue computing BMI normally
    w["weight_kg"] = pd.to_numeric(w["Value"], errors="coerce")
    h["height_raw"] = pd.to_numeric(h["Value"], errors="coerce")

    h["height_m"] = h["height_raw"].apply(
        lambda x: x / 100.0 if pd.notnull(x) and x > 3 else x
    )
    h = h.dropna(subset=["height_m"])

    # Last height per admission
    h_latest = (
        h.sort_values(["PatientId", "VisitId", "StartDateTime"])
         .groupby(["PatientId", "VisitId"], as_index=False)
         .tail(1)[["PatientId", "VisitId", "height_m"]]
    )

    # Join height to weights
    merged = w.merge(
        h_latest,
        on=["PatientId", "VisitId"],
        how="inner",
        validate="many_to_one",
    ).dropna(subset=["weight_kg", "height_m"])

    merged["BMI"] = merged["weight_kg"] / (merged["height_m"] ** 2)
    merged = merged[merged["BMI"].between(5, 250)]

    bmi_rows = merged.apply(
        lambda row: {
            "PatientId":     row["PatientId"],
            "VisitId":       row["VisitId"],
            "ConceptName":   bmi_concept,
            "StartDateTime": row["StartDateTime"],
            "EndDateTime":   row["EndDateTime"],
            "Value":         row["BMI"],
        },
        axis=1,
    ).tolist()

    return pd.DataFrame(bmi_rows)

def add_egfr_rows_from_events(
    events_df: pd.DataFrame,
    visits_relevant: pd.DataFrame,
    creatinine_concept: str = "CREATININE_SERUM_MEASURE",
    egfr_concept: str = "EGFR_MEASURE"
) -> pd.DataFrame:
    """
    Compute MDRD eGFR rows from serum creatinine events.

    - Does NOT skip if eGFR already exists
    - Uses creatinine serum in mg/dL
    - No race correction
    - Gender pulled from visits_relevant['gender']
    - Age computed from visit start vs birthdate
    - Outputs only newly computed eGFR rows
    """

    df = events_df.copy()
    df["StartDateTime"] = pd.to_datetime(df["StartDateTime"], utc=True)
    df["EndDateTime"] = pd.to_datetime(df["EndDateTime"], utc=True)

    # --- Filter creatinine rows ---
    cr = df[df["ConceptName"] == creatinine_concept].copy()
    if cr.empty:
        return pd.DataFrame(columns=[
            "PatientId", "VisitId", "ConceptName",
            "StartDateTime", "EndDateTime", "Value"
        ])

    cr["creatinine"] = pd.to_numeric(cr["Value"], errors="coerce")
    cr = cr.dropna(subset=["creatinine"])
    cr = cr[cr["creatinine"] > 0]

    # --- Join visit metadata ---
    visits = visits_relevant[[
        "person_id", "visit_id", "gender", "birth_datetime", "start_datetime"
    ]].copy()

    visits["birth_datetime"] = pd.to_datetime(visits["birth_datetime"], utc=True)
    visits["start_datetime"] = pd.to_datetime(visits["start_datetime"], utc=True)

    cr = cr.merge(
        visits,
        left_on=["PatientId", "VisitId"],
        right_on=["person_id", "visit_id"],
        how="inner",
        validate="many_to_one"
    )
    cr = cr.drop(columns=["person_id", "visit_id"])

    # --- Compute age at measurement ---
    cr["age"] = (
        (cr["StartDateTime"] - cr["birth_datetime"])
        .dt.total_seconds() / (365.25 * 24 * 3600)
    )

    cr = cr[(cr["age"] >= 18) & (cr["age"] <= 120)]

    # --- Gender coefficient ---
    cr["sex_coeff"] = cr["gender"].map({
        "FEMALE": 0.742,
        "MALE": 1.0
    })

    cr = cr.dropna(subset=["sex_coeff"])

    # --- MDRD calculation ---
    cr["egfr"] = (
        175
        * (cr["creatinine"] ** -1.154)
        * (cr["age"] ** -0.203)
        * cr["sex_coeff"]
    )

    # --- Clinical bounds ---
    cr = cr[cr["egfr"].between(1, 200)]

    # --- Emit rows ---
    egfr_rows = cr.apply(
        lambda row: {
            "PatientId": row["PatientId"],
            "VisitId": row["VisitId"],
            "ConceptName": egfr_concept,
            "StartDateTime": row["StartDateTime"],
            "EndDateTime": row["EndDateTime"],
            "Value": row["egfr"]
        },
        axis=1
    ).tolist()

    return pd.DataFrame(egfr_rows)

def add_base_glucose_rows_from_events(
    events_df: pd.DataFrame,
    hba1c_concept: str = "HEMOGLOBIN-A1C_MEASURE",
    gtt_concept: str = "GTT_MEASURE",
    base_concept: str = "BASE_GLUCOSE_MEASURE",
) -> pd.DataFrame:
    """
    Create BASE_GLUCOSE_MEASURE rows per admission using HBA1C and/or GTT.

    Logic per (PatientId, VisitId):
      - If BASE_GLUCOSE_MEASURE already exists -> skip admission.
      - Else if HBA1C exists:
            * take the latest HBA1C event in that admission
            * convert HbA1c (%) to estimated average glucose (mg/dL) with
              eAG = 28.7 * HbA1c - 46.7 (only if 3 <= HbA1c <= 18)
            * if HbA1c outside range, just copy HbA1c value as-is.
      - Else if GTT exists:
            * take the latest GTT event in that admission
            * use its numeric value directly.
      - Else: no row added.

    Returns a DataFrame with:
      PatientId, VisitId, ConceptName, StartDateTime, EndDateTime, Value
    """

    df = events_df.copy()
    df["StartDateTime"] = pd.to_datetime(df["StartDateTime"], utc=True)
    df["EndDateTime"]   = pd.to_datetime(df["EndDateTime"], utc=True)

    # 1. Identify admissions that already have BASE_GLUCOSE_MEASURE
    base_exists = (
        df[df["ConceptName"] == base_concept]
        .groupby(["PatientId", "VisitId"])
        .size()
        .reset_index()[["PatientId", "VisitId"]]
    )
    base_exists["skip"] = True

    # 2. Extract candidate events
    hba = df[df["ConceptName"] == hba1c_concept].copy()
    gtt = df[df["ConceptName"] == gtt_concept].copy()

    # Attach skip flags
    hba = hba.merge(base_exists, on=["PatientId", "VisitId"], how="left")
    gtt = gtt.merge(base_exists, on=["PatientId", "VisitId"], how="left")

    # Remove admissions where BASE_GLUC already exists
    hba = hba[hba["skip"].isna()]
    gtt = gtt[gtt["skip"].isna()]

    # Ensure numeric values
    hba["hba1c_val"] = pd.to_numeric(hba["Value"], errors="coerce")
    gtt["gtt_val"]   = pd.to_numeric(gtt["Value"], errors="coerce")

    # Keep latest HBA1C and GTT per admission (if any)
    hba_latest = (
        hba.dropna(subset=["hba1c_val"])
           .sort_values(["PatientId", "VisitId", "StartDateTime"])
           .groupby(["PatientId", "VisitId"], as_index=False)
           .tail(1)[["PatientId", "VisitId", "StartDateTime", "EndDateTime", "hba1c_val"]]
    )

    gtt_latest = (
        gtt.dropna(subset=["gtt_val"])
           .sort_values(["PatientId", "VisitId", "StartDateTime"])
           .groupby(["PatientId", "VisitId"], as_index=False)
           .tail(1)[["PatientId", "VisitId", "StartDateTime", "EndDateTime", "gtt_val"]]
    )

    # 3. Decide per admission: use HBA1C if present, else GTT
    # First, mark where HBA1C is available
    hba_latest["use_hba1c"] = True

    # Left join GTT so we know if there is a fallback
    merged_sources = pd.merge(
        hba_latest,
        gtt_latest,
        on=["PatientId", "VisitId"],
        how="outer",
        suffixes=("_hba", "_gtt"),
    )

    # Build rows
    rows = []

    for _, row in merged_sources.iterrows():
        pid = row["PatientId"]
        vid = row["VisitId"]

        value = None
        start = None
        end = None
        use_source = False

        # Primary: HBA1C
        hba_val = row.get("hba1c_val")
        if pd.notnull(hba_val):
            # Use only physiologic HbA1c values
            if 3.5 <= hba_val <= 15:
                start = row["StartDateTime_hba"]
                end   = row["EndDateTime_hba"]
                value =   28.7 * hba_val - 46.7  # eAG mg/dL
                use_source = True

        # Fallback: GTT if HbA1c not used or invalid
        if not use_source:
            gtt_val = row.get("gtt_val")
            if pd.notnull(gtt_val):
                value = gtt_val
                start = row["StartDateTime_gtt"]
                end   = row["EndDateTime_gtt"]

        if value is None or pd.isnull(start):
            continue  # nothing usable for this admission

        rows.append({
            "PatientId":     pid,
            "VisitId":       vid,
            "ConceptName":   base_concept,
            "StartDateTime": start,
            "EndDateTime":   end,
            "Value":         value,
        })

    if not rows:
        return pd.DataFrame(columns=[
            "PatientId", "VisitId", "ConceptName",
            "StartDateTime", "EndDateTime", "Value"
        ])

    return pd.DataFrame(rows)

def add_acidosis_rows_from_events(
    events_df: pd.DataFrame,
    ph_concept: str = "PH_MEASURE",
    bicarb_concept: str = "BICARBONATE_MEASURE",
    insulin_iv_concept: str = "INSULIN_IV_DOSAGE",
    ketoacidosis_concept: str = "KETOACIDOSIS",
    obs_concept: str = "KETOACIDOSIS_OBS",
    ph_threshold: float = 7.3,
    bicarb_threshold: float = 10.0,
    bicarb_window_h: float = 24.0,
    insulin_window_h: float = 6.0,
    use_observation: bool = True,
) -> pd.DataFrame:
    """
    Purpose: Emit KETOACIDOSIS event rows using a two-path strategy.
    Method:
      Observation path (use_observation=True):
        Re-emit each ICD-coded row (ConceptName == obs_concept, i.e.
        KETOACIDOSIS_OBS) as a KETOACIDOSIS event at its original timestamp.
      Derived path:
        For visits that have at least one KETOACIDOSIS_OBS row (ICD gate) AND
        satisfy the measurement rule:
          - PH_MEASURE <= ph_threshold
          - BICARBONATE_MEASURE <= bicarb_threshold within +-bicarb_window_h
          - INSULIN_IV_DOSAGE > 0 within +-insulin_window_h
        Emit one KETOACIDOSIS event per triggering pH draw.

    Args:
        events_df (pd.DataFrame): Temporal events.
        ph_concept (str): pH measurement concept name.
        bicarb_concept (str): Bicarbonate measurement concept name.
        insulin_iv_concept (str): IV insulin dosage concept name.
        ketoacidosis_concept (str): Output concept name for all emitted rows.
        obs_concept (str): ICD-coded input concept that gates both paths.
        ph_threshold (float): Max pH to trigger (<=).
        bicarb_threshold (float): Max bicarbonate mEq/L to trigger (<=).
        bicarb_window_h (float): Time window around pH for bicarbonate match.
        insulin_window_h (float): Time window around pH for insulin match.
        use_observation (bool): If True, re-emit ICD observation rows.

    Returns:
        pd.DataFrame: Rows with PatientId, VisitId, ConceptName,
            StartDateTime, EndDateTime, Value.
    """
    _empty = pd.DataFrame(columns=[
        "PatientId", "VisitId", "ConceptName", "StartDateTime", "EndDateTime", "Value"
    ])

    df = events_df.copy()
    df["StartDateTime"] = pd.to_datetime(df["StartDateTime"], errors="coerce", utc=True)
    df["EndDateTime"]   = pd.to_datetime(df["EndDateTime"],   errors="coerce", utc=True)
    df["Value_num"]     = pd.to_numeric(df["Value"], errors="coerce")

    out_rows = []

    # ICD-coded visits that gate both paths
    keto_obs = df[df["ConceptName"] == obs_concept]
    if keto_obs.empty:
        return _empty
    gated_keys = set(zip(keto_obs["PatientId"].astype(str), keto_obs["VisitId"].astype(str)))

    # Observation path: re-emit KETOACIDOSIS_OBS rows as KETOACIDOSIS
    if use_observation:
        for _, row in keto_obs.iterrows():
            out_rows.append({
                "PatientId":     row["PatientId"],
                "VisitId":       row["VisitId"],
                "ConceptName":   ketoacidosis_concept,
                "StartDateTime": row["StartDateTime"],
                "EndDateTime":   row["EndDateTime"],
                "Value":         True,
            })

    # Derived path: pH + bicarb + insulin rule -- no ICD gate required,
    # the triple measurement condition is strong enough on its own.
    ph = df[df["ConceptName"] == ph_concept].dropna(
        subset=["PatientId", "VisitId", "StartDateTime", "EndDateTime", "Value_num"]
    )
    ph = ph[ph["Value_num"] <= ph_threshold].copy()

    bic = df[df["ConceptName"] == bicarb_concept].dropna(
        subset=["PatientId", "VisitId", "StartDateTime", "Value_num"]
    )
    bic = bic[bic["Value_num"] <= bicarb_threshold].copy()

    ins = df[df["ConceptName"] == insulin_iv_concept].dropna(
        subset=["PatientId", "VisitId", "StartDateTime"]
    )
    ins = ins[pd.to_numeric(ins["Value"], errors="coerce") > 0].copy()

    if not ph.empty and not bic.empty and not ins.empty:
        bwin = pd.Timedelta(hours=bicarb_window_h)
        iwin = pd.Timedelta(hours=insulin_window_h)

        ph_bic = ph.merge(
            bic[["PatientId", "VisitId", "StartDateTime"]].rename(columns={"StartDateTime": "BIC_Start"}),
            on=["PatientId", "VisitId"], how="inner",
        )
        ph_bic = ph_bic[
            (ph_bic["BIC_Start"] >= ph_bic["StartDateTime"] - bwin) &
            (ph_bic["BIC_Start"] <= ph_bic["StartDateTime"] + bwin)
        ]
        ph_ok = ph_bic[["PatientId", "VisitId", "StartDateTime", "EndDateTime"]].drop_duplicates()

        ph_ins = ph_ok.merge(
            ins[["PatientId", "VisitId", "StartDateTime"]].rename(columns={"StartDateTime": "INS_Start"}),
            on=["PatientId", "VisitId"], how="inner",
        )
        ph_ins = ph_ins[
            (ph_ins["INS_Start"] >= ph_ins["StartDateTime"] - iwin) &
            (ph_ins["INS_Start"] <= ph_ins["StartDateTime"] + iwin)
        ]
        for _, row in ph_ins[["PatientId", "VisitId", "StartDateTime", "EndDateTime"]].drop_duplicates().iterrows():
            out_rows.append({
                "PatientId":     row["PatientId"],
                "VisitId":       row["VisitId"],
                "ConceptName":   ketoacidosis_concept,
                "StartDateTime": row["StartDateTime"],
                "EndDateTime":   row["EndDateTime"],
                "Value":         True,
            })

    if not out_rows:
        return _empty
    return pd.DataFrame(out_rows)
def add_infection_rows_from_events(
    events_df: pd.DataFrame,
    infection_concept: str = "INFECTION",

    # measurement concepts
    temp_concept: str = "BODY_TEMPERATURE_MEASURE",
    wbc_concept: str = "INFECTION_WBC_MEASUREMENT",
    neut_concept: str = "NEUTROPHILS_MEASURE",

    # diagnosis concepts
    diagnosis_concepts=("BLOOD_CULTURE", "URINE_CULTURE", "CHEST_XRAY"),

    # treatment concepts
    antibiotic_concepts=("ANTIBIOTIC_IV_BITZUA", "ANTIBIOTIC_PO_BITZUA"),

    # thresholds
    temp_hi: float = 37.8,
    temp_prev_max: float = 37.7,
    wbc_hi: float = 11,
    wbc_lo: float = 4,
    neut_lo: float = 1.5,

    # windows
    baseline_window_h: float = 48,       # "after 48h of normal if measured"
    diagnosis_gap_h: float = 24,         # diagnosis within ±24h of trigger
    episode_collapse_h: float = 72,      # don't emit events within 72h of an accepted event
    abx_after_ref_h: float = 72          # require at least 1 abx after diagnosis and within 72h of ref_time
) -> pd.DataFrame:
    """
    Create INFECTION event rows if ALL are met for an episode within an admission (PatientId, VisitId):

    Measurement trigger (ANY ONE):
      A) Fever onset:
         - at least 2 BODY_TEMPERATURE_MEASURE values >= 37.8
         - and all temps in the previous 48h (if existed) were <= 37.7
         - reference time = first temp >= 37.8 that satisfies the above

      B) WBC high:
         - INFECTION_WBC_MEASUREMENT >= 11
         - and WBC in the previous 48h (if existed) were between 4 and 11 (inclusive)

      C) WBC low:
         - INFECTION_WBC_MEASUREMENT <= 4
         - and WBC in the previous 48h (if existed) were between 4 and 11 (inclusive)

      D) Neutropenia:
         - NEUTROPHILS_MEASURE <= 1.5
         - and neutrophils in the previous 48h (if existed) were > 1.5

    Diagnosis evidence (required):
      - at least one of BLOOD_CULTURE / URINE_CULTURE / CHEST_XRAY within ±24h of the reference time
      - we use the LATEST diagnosis row in that window as the anchor timestamp

    Treatment (required):
      - at least one antibiotic administration AFTER that diagnosis anchor
      - and within 72h of the reference time (to tie it to the same episode)

    Multiple episodes per admission:
      - We evaluate multiple candidate triggers
      - We only "collapse" triggers that are within 72h of a previously ACCEPTED infection event
      - Output can contain multiple INFECTION rows per admission

    Output:
      - One row per accepted episode
      - Start/End times copied from the anchor diagnosis row (latest within ±24h)
      - Value = "True"
    """

    df = events_df.copy()
    df["StartDateTime"] = pd.to_datetime(df["StartDateTime"], errors="coerce", utc=True)
    df["EndDateTime"] = pd.to_datetime(df["EndDateTime"], errors="coerce", utc=True)
    df["ValueNum"] = pd.to_numeric(df["Value"], errors="coerce")

    required_cols = {"PatientId", "VisitId", "ConceptName", "StartDateTime", "EndDateTime", "Value"}
    missing = required_cols - set(df.columns)
    if missing:
        raise KeyError(f"events_df missing required columns: {missing}")

    out_rows = []
    base_win = timedelta(hours=baseline_window_h)
    diag_win = timedelta(hours=diagnosis_gap_h)
    collapse_win = timedelta(hours=episode_collapse_h)
    abx_win = timedelta(hours=abx_after_ref_h)

    for (pid, vid), adm in df.groupby(["PatientId", "VisitId"], sort=False):
        adm = adm.dropna(subset=["StartDateTime"]).sort_values("StartDateTime")

        # ---------- collect candidate reference times (measurement triggers) ----------
        ref_times = []

        # A) Fever onset trigger
        temps = adm[adm["ConceptName"] == temp_concept].dropna(subset=["ValueNum"])
        if not temps.empty:
            temps = temps.sort_values("StartDateTime")
            # We require >=2 temps >= temp_hi occurring at/after the reference time
            hi_mask = temps["ValueNum"] >= temp_hi
            if hi_mask.any():
                # Precompute cumulative count of highs from each index forward
                # Simple approach: for each candidate t0, count highs in temps >= t0
                for _, row in temps[hi_mask].iterrows():
                    t0 = row["StartDateTime"]
                    prev48 = temps[(temps["StartDateTime"] < t0) & (temps["StartDateTime"] >= t0 - base_win)]
                    if (prev48.empty) or (prev48["ValueNum"] <= temp_prev_max).all():
                        after = temps[temps["StartDateTime"] >= t0]
                        if (after["ValueNum"] >= temp_hi).sum() >= 2:
                            ref_times.append(t0)

        # B/C) WBC triggers
        wbc = adm[adm["ConceptName"] == wbc_concept].dropna(subset=["ValueNum"])
        if not wbc.empty:
            wbc = wbc.sort_values("StartDateTime")
            for _, row in wbc.iterrows():
                t0 = row["StartDateTime"]
                prev48 = wbc[(wbc["StartDateTime"] < t0) & (wbc["StartDateTime"] >= t0 - base_win)]
                # require baseline window exists AND is "normal" 4..11 (per your spec)
                if (not prev48.empty) and prev48["ValueNum"].between(wbc_lo, wbc_hi, inclusive="both").all():
                    if row["ValueNum"] >= wbc_hi or row["ValueNum"] <= wbc_lo:
                        ref_times.append(t0)

        # D) Neutrophils trigger
        neut = adm[adm["ConceptName"] == neut_concept].dropna(subset=["ValueNum"])
        if not neut.empty:
            neut = neut.sort_values("StartDateTime")
            for _, row in neut.iterrows():
                t0 = row["StartDateTime"]
                prev48 = neut[(neut["StartDateTime"] < t0) & (neut["StartDateTime"] >= t0 - base_win)]
                # require baseline window exists AND is > 1.5 (per your spec)
                if (not prev48.empty) and (prev48["ValueNum"] > neut_lo).all():
                    if row["ValueNum"] <= neut_lo:
                        ref_times.append(t0)

        if not ref_times:
            continue

        ref_times = sorted(set(ref_times))

        # ---------- episode evaluation with post-accept collapse ----------
        last_accepted_ref_time = None

        for ref_time in ref_times:
            if last_accepted_ref_time is not None and (ref_time - last_accepted_ref_time) < collapse_win:
                continue

            # Diagnosis within ±24h of ref_time; use the latest diagnosis row in that window
            diag = adm[
                adm["ConceptName"].isin(diagnosis_concepts) &
                adm["StartDateTime"].between(ref_time - diag_win, ref_time + diag_win)
            ].copy()

            if diag.empty:
                continue

            diag = diag.sort_values("StartDateTime")
            diag_row = diag.iloc[-1]  # latest within window

            # Antibiotic after diagnosis and within 72h of ref_time
            abx = adm[
                adm["ConceptName"].isin(antibiotic_concepts) &
                (adm["StartDateTime"] > diag_row["StartDateTime"]) &
                (adm["StartDateTime"] <= ref_time + abx_win)
            ]

            if abx.empty:
                continue

            # Emit infection event anchored to diagnosis row time
            out_rows.append({
                "PatientId": pid,
                "VisitId": vid,
                "ConceptName": infection_concept,
                "StartDateTime": diag_row["StartDateTime"],
                "EndDateTime": diag_row["EndDateTime"],
                "Value": "True"
            })

            # Collapse only after acceptance
            last_accepted_ref_time = ref_time

    return pd.DataFrame(out_rows, columns=["PatientId", "VisitId", "ConceptName", "StartDateTime", "EndDateTime", "Value"])


def add_hyperosmolality_rows_from_events(
    events_df: pd.DataFrame,
    glucose_concept: str = "GLUCOSE_MEASURE",
    sodium_concept: str = "SODIUM_MEASURE",
    acidosis_concept: str = "ACIDOSIS",
    hyperosmolality_concept: str = "HYPEROSMOLALITY",
    glucose_threshold: float = 600.0,
    osm_threshold: float = 320.0,
    na_window_h: float = 6.0,
    acidosis_window_h: float = 6.0,
    use_observation: bool = True,
    obs_concept: str = None,
) -> pd.DataFrame:
    """
    Purpose: Emit HYPEROSMOLALITY event rows using a two-path strategy.
    Method:
      Derived path (HHS-like rule, always active):
        For each GLUCOSE_MEASURE >= glucose_threshold in an admission:
          - Find the nearest SODIUM_MEASURE within ±na_window_h hours.
          - Compute effective_osmolality = 2*Na + Glucose/18.
          - If eff_osm >= osm_threshold AND no ACIDOSIS event within
            ±acidosis_window_h of the glucose draw -> emit event.
          - Timestamped at the triggering glucose measurement.
      Observation path (use_observation=True):
        Also emit a True row for every row in events_df where
        ConceptName == hyperosmolality_concept (direct ICD/observation mapping).

    Args:
        events_df (pd.DataFrame): Temporal events with PatientId, VisitId,
            ConceptName, StartDateTime, EndDateTime, Value.
        glucose_concept (str): Concept name for glucose measurements.
        sodium_concept (str): Concept name for serum sodium measurements.
        acidosis_concept (str): Concept name for acidosis events (DKA gate).
        hyperosmolality_concept (str): Output concept name.
        glucose_threshold (float): Minimum glucose mg/dL to trigger.
        osm_threshold (float): Minimum effective osmolality mOsm/kg to trigger.
        na_window_h (float): Max hours between glucose and sodium draws.
        acidosis_window_h (float): Exclusion window hours around acidosis events.
        use_observation (bool): If True, also re-emit direct observations of concept.

    Returns:
        pd.DataFrame: Rows with PatientId, VisitId, ConceptName,
            StartDateTime, EndDateTime, Value.
    """
    obs_concept = obs_concept or f"{hyperosmolality_concept}_OBS"
    df = events_df.copy()
    df["StartDateTime"] = pd.to_datetime(df["StartDateTime"], errors="coerce", utc=True)
    df["EndDateTime"]   = pd.to_datetime(df["EndDateTime"],   errors="coerce", utc=True)
    df["ValueNum"]      = pd.to_numeric(df["Value"], errors="coerce")

    out_rows = []

    # Observation path: ICD-coded rows (_OBS) re-emitted as HYPEROSMOLALITY
    if use_observation:
        for _, row in df[df["ConceptName"] == obs_concept].iterrows():
            out_rows.append({
                "PatientId":     row["PatientId"],
                "VisitId":       row["VisitId"],
                "ConceptName":   hyperosmolality_concept,
                "StartDateTime": row["StartDateTime"],
                "EndDateTime":   row["EndDateTime"],
                "Value":         True,
            })

    # Derived path: HHS-like rule per admission
    na_win = pd.Timedelta(hours=na_window_h)
    ac_win = pd.Timedelta(hours=acidosis_window_h)

    for (pid, vid), adm in df.groupby(["PatientId", "VisitId"], sort=False):
        adm = adm.dropna(subset=["StartDateTime"]).sort_values("StartDateTime")

        glu = adm[(adm["ConceptName"] == glucose_concept) & (adm["ValueNum"] >= glucose_threshold)]
        if glu.empty:
            continue

        na = adm[adm["ConceptName"] == sodium_concept].dropna(subset=["ValueNum"])
        if na.empty:
            continue

        acidosis = adm[adm["ConceptName"] == acidosis_concept]

        for _, g_row in glu.iterrows():
            t0       = g_row["StartDateTime"]
            gluc_val = g_row["ValueNum"]

            na_nearby = na[
                (na["StartDateTime"] >= t0 - na_win) &
                (na["StartDateTime"] <= t0 + na_win)
            ].copy()
            if na_nearby.empty:
                continue

            # Closest Na measurement by absolute time distance
            na_nearby["_dist"] = (na_nearby["StartDateTime"] - t0).abs()
            na_val = na_nearby.loc[na_nearby["_dist"].idxmin(), "ValueNum"]

            if 2 * na_val + gluc_val / 18.0 < osm_threshold:
                continue

            # Gate: skip if ACIDOSIS event is nearby (likely DKA, not HHS)
            if not acidosis.empty:
                if not acidosis[
                    (acidosis["StartDateTime"] >= t0 - ac_win) &
                    (acidosis["StartDateTime"] <= t0 + ac_win)
                ].empty:
                    continue

            out_rows.append({
                "PatientId":     pid,
                "VisitId":       vid,
                "ConceptName":   hyperosmolality_concept,
                "StartDateTime": t0,
                "EndDateTime":   g_row["EndDateTime"],
                "Value":         True,
            })

    if not out_rows:
        return pd.DataFrame(columns=[
            "PatientId", "VisitId", "ConceptName",
            "StartDateTime", "EndDateTime", "Value",
        ])
    return pd.DataFrame(out_rows)


def add_cardiovascular_disorder_rows_from_events(
    events_df: pd.DataFrame,
    troponin_concept: str = "TROPONIN_MEASURE",
    cv_gate_concept: str = "CARDIOVASCULAR_DISORDER_OBS",
    cardiovascular_concept: str = "CARDIOVASCULAR_DISORDER",
    troponin_threshold: float = 600.0,
    use_observation: bool = True,
) -> pd.DataFrame:
    """
    Purpose: Emit CARDIOVASCULAR_DISORDER event rows using a two-path strategy.
    Method:
      Observation path (use_observation=True):
        Re-emit each ICD/observation-coded row (ConceptName == cv_gate_concept,
        i.e. CARDIOVASCULAR_DISORDER_OBS) as a CARDIOVASCULAR_DISORDER event,
        preserving its original timestamp.
      Derived path (always active):
        For each admission where cv_gate_concept rows exist (ICD gate) AND a
        TROPONIN_MEASURE >= troponin_threshold is found in the same admission:
          - Emit one CARDIOVASCULAR_DISORDER event per qualifying troponin draw,
            timestamped at that draw.

    Args:
        events_df (pd.DataFrame): Temporal events with PatientId, VisitId,
            ConceptName, StartDateTime, EndDateTime, Value.
        troponin_concept (str): Concept name for troponin measurements.
        cv_gate_concept (str): Input concept that identifies ICD-coded CV rows
            (default CARDIOVASCULAR_DISORDER_OBS, produced by the mapping JSONs).
        cardiovascular_concept (str): Output concept name for all emitted rows.
        troponin_threshold (float): Minimum troponin value to trigger derived path.
        use_observation (bool): If True, also re-emit ICD observation rows.

    Returns:
        pd.DataFrame: Rows with PatientId, VisitId, ConceptName,
            StartDateTime, EndDateTime, Value.
    """
    df = events_df.copy()
    df["StartDateTime"] = pd.to_datetime(df["StartDateTime"], errors="coerce", utc=True)
    df["EndDateTime"]   = pd.to_datetime(df["EndDateTime"],   errors="coerce", utc=True)
    df["ValueNum"]      = pd.to_numeric(df["Value"], errors="coerce")

    out_rows = []

    # Observation path: ICD-coded rows (_OBS) re-emitted as CARDIOVASCULAR_DISORDER
    if use_observation:
        for _, row in df[df["ConceptName"] == cv_gate_concept].iterrows():
            out_rows.append({
                "PatientId":     row["PatientId"],
                "VisitId":       row["VisitId"],
                "ConceptName":   cardiovascular_concept,
                "StartDateTime": row["StartDateTime"],
                "EndDateTime":   row["EndDateTime"],
                "Value":         True,
            })

    # Derived path: visits with any ICD gate row + troponin >= threshold
    cv_obs = df[df["ConceptName"] == cv_gate_concept]
    if not cv_obs.empty:
        cv_keys = set(
            zip(cv_obs["PatientId"].astype(str), cv_obs["VisitId"].astype(str))
        )

        trop = df[
            (df["ConceptName"] == troponin_concept) &
            (df["ValueNum"] >= troponin_threshold)
        ].copy()

        if not trop.empty:
            trop["_key"] = list(zip(trop["PatientId"].astype(str), trop["VisitId"].astype(str)))
            for _, row in trop[trop["_key"].isin(cv_keys)].iterrows():
                out_rows.append({
                    "PatientId":     row["PatientId"],
                    "VisitId":       row["VisitId"],
                    "ConceptName":   cardiovascular_concept,
                    "StartDateTime": row["StartDateTime"],
                    "EndDateTime":   row["EndDateTime"],
                    "Value":         True,
                })

    if not out_rows:
        return pd.DataFrame(columns=[
            "PatientId", "VisitId", "ConceptName",
            "StartDateTime", "EndDateTime", "Value",
        ])
    return pd.DataFrame(out_rows)

def add_special_rows(events_df: pd.DataFrame,
                     visits_df: pd.DataFrame,
                     id_column: str = "person_id",
                     visit_column: str = "visit_id") -> pd.DataFrame:
    """
    Add synthetic rows:
      - ADMISSION / RELEASE / DEATH
      - MEAL rows
      - BMI_MEASURE rows
      - BASE_GLUCOSE_MEASURE rows (from HBA1C / GTT)
    """

    # 1) Admission / Release / Death
    ar_rows = build_admission_release_rows(
        visits_df,
        id_column=id_column,
        visit_column=visit_column,
    )

    # 2) Meals
    meal_rows = add_fake_meal_rows(
        visits_df,
        id_column=id_column,
        visit_column=visit_column,
    )

    # 3) BMI from temporal WEIGHT/HEIGHT (skips admissions with BMI already)
    bmi_rows = add_bmi_rows_from_events(events_df)
    
    # 4) GFR-MDRD rows
    egfr_rows = add_egfr_rows_from_events( events_df, visits_df)

    # 5) Base glucose from HBA1C / GTT (skips admissions with BASE already)
    base_gluc_rows = add_base_glucose_rows_from_events(events_df)
    
    # 6) Acidosis event treated with insulin IV
    acidosis_rows = add_acidosis_rows_from_events(events_df)
    
    # 7) Infection rows
    infection_rows = add_infection_rows_from_events(events_df)

    # 8) Hyperosmolality (HHS-like) from glucose + sodium
    hyperosmolality_rows = add_hyperosmolality_rows_from_events(events_df)

    # 9) Cardiovascular disorder from troponin (gated by CV ICD observation)
    cardiovascular_rows = add_cardiovascular_disorder_rows_from_events(events_df)

    _parts = [ar_rows, meal_rows, bmi_rows, egfr_rows,
              base_gluc_rows, acidosis_rows, infection_rows,
              hyperosmolality_rows, cardiovascular_rows]
    specials = pd.concat(
        [p for p in _parts if not p.empty],
        ignore_index=True,
        sort=False,
    )

    return specials

def map_events_partition(pdf: pd.DataFrame,
                         table_name: str,
                         table_specs: dict) -> pd.DataFrame:
    """
    Map one Pandas partition from a source table to mediator events.

    Input partition columns (shared across tables):
      person_id, visit_id, concept_id, concept_name,
      start_datetime, end_datetime, value?, unit?, drug_type_category?, route?, quantity?

    Output columns:
      PatientId, VisitId, ConceptName, StartDateTime, EndDateTime, Value
    """
    if pdf.empty or not table_specs:
        return pd.DataFrame(columns=[
            "PatientId", "VisitId", "ConceptName",
            "StartDateTime", "EndDateTime", "Value",
        ])

    # Ensure datetime
    pdf = pdf.copy()
    pdf["start_datetime"] = pd.to_datetime(pdf["start_datetime"], utc=True)
    pdf["end_datetime"] = pd.to_datetime(pdf["end_datetime"], errors="coerce", utc=True)

    # We will use these as output
    rows = []

    for mediator_concept, rules in table_specs.items():
        total_matched_for_concept = 0

        for rule in rules:
            conds = rule["conditions"]
            mask = pd.Series(True, index=pdf.index)

            for cond in conds:
                col = cond["column"]
                op  = cond["op"]

                if col not in pdf.columns:
                    mask &= False
                    continue

                # IDs (concept_id) ? compare as string, no need for lower()
                if op == "in_ids":
                    ids = set(cond["ids"])
                    mask &= pdf[col].astype(str).isin(ids)

                # Textual equality (route, drug_type_category etc.)
                elif op == "eq":
                    vals = [str(v).lower() for v in cond["values"]]
                    mask &= pdf[col].astype(str).str.lower().isin(vals)

                else:
                    raise ValueError(f"Unknown op '{op}' in index_specs")

            matched = pdf[mask]
            if matched.empty:
                continue

            total_matched_for_concept += len(matched)

            # Decide which column is the "Value" here
            if table_name == "measurement":
                value_col = "value"
            elif table_name == "drug_exposure":
                # You asked that quantity maps to Value
                value_col = "quantity"
            else:
                value_col = None

            for _, row in matched.iterrows():
                start = row["start_datetime"]
                end   = row["end_datetime"]
                if pd.isna(end) or end < start:
                    end = start + pd.Timedelta(seconds=1)

                if value_col is not None and value_col in row and pd.notna(row[value_col]):
                    value = row[value_col]
                else:
                    value = True

                rows.append({
                    "PatientId":     str(row["person_id"]),
                    "VisitId":       str(row["visit_id"]),
                    "ConceptName":   mediator_concept,
                    "StartDateTime": start,
                    "EndDateTime":   end,
                    "Value":         value,
                })

    if not rows:
        return pd.DataFrame(columns=[
            "PatientId", "VisitId", "ConceptName",
            "StartDateTime", "EndDateTime", "Value",
        ])

    return pd.DataFrame(rows)

def build_mediator_events_parquet(
    index_specs: dict,
    data_dir: str,
    visits_master_path: str = "visits_master.csv",
    events_parquet_path: str = "mediator_events",
):
    """
    Purpose: Map raw source tables to mediator-format events and write as Parquet.
    Method:
      Phase 1 — per-table, per-partition:
        For each source table (measurement, clinical_events, drug_exposure):
          - Load one Dask partition at a time with scheduler=synchronous.
          - Filter to relevant patients only.
          - Apply index_specs mapping via map_events_partition.
          - Write mapped rows to _tmp_mediator_events/<table>/part_XXXXX.parquet.
          - Resume markers (.done_XXXXX) allow restart after crash.
      Phase 2 — assembly:
        - Read all tmp parquets into pandas (mapped events are much smaller than raw).
        - Merge visit metadata (AdmissionStart, AdmissionEnd, relevant_admission).
        - Write final Parquet dataset to events_parquet_path.
        - Delete _tmp_ folder.

    Args:
        index_specs (dict): Mapping of table_name -> {concept -> [rules]}.
        data_dir (str): Root data directory (unused directly; get_data handles paths).
        visits_master_path (str): Path to visits_master.csv.
        events_parquet_path (str): Output parquet directory.
    """
    import shutil
    _TMP = Path("_tmp_mediator_events")
    _TMP.mkdir(parents=True, exist_ok=True)

    # ── visits master ─────────────────────────────────────────────────────────
    visits = pd.read_csv(visits_master_path, low_memory=False)
    visits["person_id"] = visits["person_id"].astype(str)
    visits["visit_id"]  = visits["visit_id"].astype(str)

    if "relevant_admission" not in visits.columns:
        visits["relevant_admission"] = (
            visits["is_inpatient"]
            & ~visits["broken_visit_id"]
            & visits["relevant_department"]
            & visits["relevant_duration"]
            & ~visits["death_within_48h"]
            & visits["adult_at_admission"]
            & visits["diabetic_condition"]
        )

    relevant_patients = set(visits.loc[visits["relevant_admission"], "person_id"])
    print(f"Patients with at least one relevant admission: {len(relevant_patients)}")

    visits_small = visits[[
        "person_id", "visit_id",
        "start_datetime", "end_datetime", "relevant_admission",
    ]].copy()
    visits_small["start_datetime"] = pd.to_datetime(visits_small["start_datetime"], utc=True)
    visits_small["end_datetime"]   = pd.to_datetime(visits_small["end_datetime"],   utc=True)

    # ── Phase 1: per-table, per-partition mapping ─────────────────────────────
    TABLE_COLUMNS = {
        "measurement": [
            "person_id", "visit_id", "concept_id", "concept_name",
            "start_datetime", "end_datetime", "value", "unit",
        ],
        "clinical_events": [
            "person_id", "visit_id", "concept_id", "concept_name",
            "start_datetime", "end_datetime",
        ],
        "drug_exposure": [
            "person_id", "visit_id", "concept_id", "concept_name",
            "start_datetime", "end_datetime",
            "drug_type_category", "route", "quantity",
        ],
    }

    for table_name, columns in TABLE_COLUMNS.items():
        _table_done = _TMP / f".done_{table_name}"
        if _table_done.exists():
            print(f"\n[{table_name}] already done, skipping")
            continue

        table_specs = index_specs.get(table_name, {})
        if not table_specs:
            print(f"\n[{table_name}] no index_specs entries, skipping")
            _table_done.touch()
            continue

        out_dir = _TMP / table_name
        out_dir.mkdir(exist_ok=True)

        print(f"\n=== Mapping table '{table_name}' from Parquet ===")
        ddf = get_data(table=table_name, columns=columns, as_dask=True)
        delayed_parts = ddf.to_delayed()
        print(f"  {len(delayed_parts)} partitions")

        for i, part in enumerate(tqdm(delayed_parts, desc=table_name, total=len(delayed_parts))):
            _part_done = out_dir / f".done_{i:05d}"
            if _part_done.exists():
                continue

            pdf = part.compute(scheduler="synchronous")
            pdf["person_id"] = pdf["person_id"].astype(str)

            # Filter to relevant patients before mapping
            pdf = pdf[pdf["person_id"].isin(relevant_patients)]
            if not pdf.empty:
                mapped = map_events_partition(pdf, table_name, table_specs)
                if not mapped.empty:
                    mapped["Value"] = mapped["Value"].astype(str)
                    mapped.to_parquet(out_dir / f"part_{i:05d}.parquet", index=False)

            _part_done.touch()

        _table_done.touch()
        print(f"[{table_name}] done")

    # ── Phase 2: assemble + merge visit metadata + write final parquet ────────
    print("\n=== Assembling final mediator_events parquet ===")

    all_parts = sorted(_TMP.rglob("part_*.parquet"))
    if not all_parts:
        raise RuntimeError("No mapped events written — check index_specs and source tables.")

    # Process in batches to avoid OOM: 82M rows cannot fit in RAM at once.
    out_path = Path(events_parquet_path)
    out_path.mkdir(parents=True, exist_ok=True)

    visits_small_renamed = visits_small.rename(columns={
        "person_id":      "PatientId",
        "visit_id":       "VisitId",
        "start_datetime": "AdmissionStart",
        "end_datetime":   "AdmissionEnd",
    })

    FILE_BATCH = 50  # parquet files per batch
    total_rows = 0
    out_idx = 0

    for batch_start in tqdm(
        range(0, len(all_parts), FILE_BATCH),
        desc="assembling",
        total=math.ceil(len(all_parts) / FILE_BATCH),
    ):
        batch_files = all_parts[batch_start : batch_start + FILE_BATCH]
        batch_pd = pd.concat(
            [pd.read_parquet(p) for p in batch_files],
            ignore_index=True,
        )
        batch_pd["PatientId"] = batch_pd["PatientId"].astype(str)
        batch_pd["VisitId"]   = batch_pd["VisitId"].astype(str)

        batch_pd = batch_pd.merge(
            visits_small_renamed,
            on=["PatientId", "VisitId"],
            how="left",
        )

        if "Value" in batch_pd.columns:
            batch_pd["Value"] = batch_pd["Value"].astype(str)

        total_rows += len(batch_pd)
        batch_pd.to_parquet(out_path / f"part_{out_idx:05d}.parquet", index=False)
        out_idx += 1
        del batch_pd

    shutil.rmtree(_TMP, ignore_errors=True)
    print(f"mediator_events written to {events_parquet_path}  ({total_rows:,} rows)")


def post_process(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply post-processing rules to mediator_input.csv.
    
    Rule 1:
        If two GLUCOSE_MEASURE events occur within 8 minutes
        (for the same person_id + visit_id), drop the earlier one.
    
    Additional rules can be added here in the future.
    """

    df = df.copy()

    # Ensure datetime is datetime
    df["StartDateTime"] = pd.to_datetime(df["StartDateTime"], errors="coerce", utc=True)

    # Sort for proper comparison
    df = df.sort_values(["PatientId", "VisitId", "StartDateTime"])

    # Mask glucose rows
    mask_glu = df["ConceptName"] == "GLUCOSE_MEASURE"
    glu = df[mask_glu].copy()

    # Compute time difference from previous glucose measurement (same visit)
    glu["prev_time"] = (
        glu.groupby(["PatientId", "VisitId"])["StartDateTime"]
        .shift(1)
    )
    glu["delta_min"] = (glu["StartDateTime"] - glu["prev_time"]).dt.total_seconds() / 60.0

    # Identify rows to drop: delta < 8 minutes
    drop_ids = set(glu.loc[glu["delta_min"] < 8, "row_id"]) if "row_id" in df.columns else \
               set(glu.loc[glu["delta_min"] < 8].index)

    # Remove the earlier duplicates
    df_clean = df.drop(index=drop_ids)

    return df_clean

def finalize_mediator_input(
    events_parquet_path: str,
    visits_master_path: str,
    output_csv: str,
    concept_windows: dict,
    batch_size_patients: int = 1000,
):
    """
    Purpose: Propagate prior concepts, add specials, clip and write mediator_input.csv.
    Method:
      - Loads visits_master (pandas) and mediator_events parquet (pandas, full read once).
      - Processes patients in batches with a tqdm progress bar.
      - Each batch is written to _tmp_finalize/batch_XXXXX.parquet with a .done marker
        so the run can resume after a crash without reprocessing completed batches.
      - Final step concatenates all batch parquets and writes output_csv.

    Args:
        events_parquet_path (str): Path to mediator_events parquet directory.
        visits_master_path (str): Path to visits_master.csv.
        output_csv (str): Output path for mediator_input.csv.
        concept_windows (dict): {concept_name -> lookback_days} for propagation.
        batch_size_patients (int): Patients per processing batch.
    """
    import shutil
    _TMP = Path("_tmp_finalize")
    _TMP.mkdir(parents=True, exist_ok=True)

    # ── Visits master ─────────────────────────────────────────────────────────
    visits = pd.read_csv(visits_master_path, low_memory=False)
    visits["person_id"] = visits["person_id"].astype(str)
    visits["visit_id"]  = visits["visit_id"].astype(str)

    if "relevant_admission" not in visits.columns:
        visits["relevant_admission"] = (
            visits["is_inpatient"]
            & ~visits["broken_visit_id"]
            & visits["relevant_department"]
            & visits["relevant_duration"]
            & ~visits["death_within_48h"]
            & visits["adult_at_admission"]
            & visits["diabetic_condition"]
        )
    visits["relevant_admission"] = visits["relevant_admission"].fillna(False).astype(bool)

    visits["start_datetime"] = (
        pd.to_datetime(visits["start_datetime"], utc=True, errors="coerce")
        .dt.tz_convert(None)
    )
    visits["end_datetime"] = (
        pd.to_datetime(visits["end_datetime"], utc=True, errors="coerce")
        .dt.tz_convert(None)
    )

    # ── Batch setup ────────────────────────────────────────────────────────────
    # Patient list comes from visits_master (already in memory) — avoids loading
    # all 82M mediator_events rows at once. Per-batch predicate pushdown reads
    # only the rows needed for each batch.
    patient_ids = sorted(
        visits.loc[visits["relevant_admission"], "person_id"].unique().tolist()
    )
    n_patients = len(patient_ids)
    n_batches  = math.ceil(n_patients / batch_size_patients)
    print(f"Processing {n_patients:,} patients in {n_batches} batches of {batch_size_patients}")

    with tqdm(total=n_batches, desc="finalize_mediator_input", unit="batch") as pbar:
        for b in range(n_batches):
            _done = _TMP / f".done_{b:05d}"
            if _done.exists():
                pbar.update(1)
                continue

            start_idx  = b * batch_size_patients
            end_idx    = min((b + 1) * batch_size_patients, n_patients)
            batch_pids = patient_ids[start_idx:end_idx]

            # Read only this batch's rows from parquet using predicate pushdown
            events_batch = pd.read_parquet(
                events_parquet_path,
                filters=[("PatientId", "in", batch_pids)],
            )
            events_batch["PatientId"] = events_batch["PatientId"].astype(str)
            events_batch["VisitId"]   = events_batch["VisitId"].astype(str)
            events_batch["StartDateTime"] = (
                pd.to_datetime(events_batch["StartDateTime"], utc=True, errors="coerce")
                .dt.tz_convert(None)
            )
            events_batch["EndDateTime"] = (
                pd.to_datetime(events_batch["EndDateTime"], utc=True, errors="coerce")
                .dt.tz_convert(None)
            )
            if events_batch.empty:
                _done.touch()
                pbar.update(1)
                continue

            # Visits for this batch
            visits_batch = visits[visits["person_id"].isin(batch_pids)].copy()
            if visits_batch.empty:
                _done.touch()
                pbar.update(1)
                continue

            # Propagate prior concepts into relevant admissions only
            prior_rows = propagate_prior_concepts(
                events_df=events_batch,
                visits_df=visits_batch,
                concept_windows=concept_windows,
                id_col_events="PatientId",
                visit_col_events="VisitId",
                id_col_visits="person_id",
                visit_col_visits="visit_id",
                start_col_visits="start_datetime",
                target_only_relevant=True,
            )
            if len(prior_rows) > 0:
                # Drop all-NA columns from prior_rows (AdmissionStart/End/relevant_admission
                # are not filled by propagate_prior_concepts) to silence FutureWarning.
                events_plus = pd.concat(
                    [events_batch, prior_rows.dropna(axis=1, how="all")],
                    ignore_index=True,
                )
            else:
                events_plus = events_batch

            # Attach relevant_admission
            relev_map = (
                visits_batch[["person_id", "visit_id", "relevant_admission"]]
                .drop_duplicates()
                .rename(columns={"person_id": "PatientId", "visit_id": "VisitId"})
            )
            relev_map["PatientId"] = relev_map["PatientId"].astype(str)
            relev_map["VisitId"]   = relev_map["VisitId"].astype(str)
            events_plus["PatientId"] = events_plus["PatientId"].astype(str)
            events_plus["VisitId"]   = events_plus["VisitId"].astype(str)

            for c in list(events_plus.columns):
                if c.startswith("relevant_admission"):
                    events_plus.drop(columns=[c], inplace=True)

            events_plus = events_plus.merge(relev_map, on=["PatientId", "VisitId"], how="left")
            events_plus["relevant_admission"] = (
                events_plus["relevant_admission"].fillna(False).astype(bool)
            )

            nan_rel = events_plus["relevant_admission"].isna().sum()
            if nan_rel > 0:
                tqdm.write(f"  [WARNING] batch {b}: {nan_rel} rows with NaN relevant_admission")

            # Keep only relevant admissions
            events_relevant = events_plus[events_plus["relevant_admission"]].copy()
            if events_relevant.empty:
                _done.touch()
                pbar.update(1)
                continue

            visits_relevant = visits_batch[visits_batch["relevant_admission"]].copy()

            # Add specials
            specials_df = add_special_rows(
                events_df=events_relevant,
                visits_df=visits_relevant,
                id_column="person_id",
                visit_column="visit_id",
            )

            # Clip to [ADMISSION .. RELEASE/DEATH]
            full_batch = pd.concat([events_relevant, specials_df], ignore_index=True)
            full_batch = filter_patient_events(full_batch)
            if full_batch.empty:
                _done.touch()
                pbar.update(1)
                continue

            # Sort & post-process
            final_batch = (
                full_batch
                .sort_values(["PatientId", "VisitId", "StartDateTime"])
                .reset_index(drop=True)
            )
            final_batch = post_process(final_batch)

            # Backfill AdmissionStart/AdmissionEnd for synthetic rows.
            # Special-row functions (acidosis, cardiovascular, etc.) emit only
            # PatientId/VisitId/ConceptName/StartDateTime/EndDateTime/Value, so
            # AdmissionStart is NaN after concat. Without this, hours_from_admission
            # is NaN and these rows are invisible in all downstream analysis.
            if "AdmissionStart" in final_batch.columns:
                _visit_times = (
                    events_relevant[["PatientId", "VisitId", "AdmissionStart", "AdmissionEnd"]]
                    .dropna(subset=["AdmissionStart"])
                    .drop_duplicates(subset=["PatientId", "VisitId"])
                    .assign(
                        PatientId=lambda d: d["PatientId"].astype(str),
                        VisitId=lambda d: d["VisitId"].astype(str),
                    )
                )
                _missing = final_batch["AdmissionStart"].isna()
                if _missing.any():
                    _filled = (
                        final_batch.loc[_missing]
                        .drop(columns=["AdmissionStart", "AdmissionEnd"])
                        .merge(_visit_times, on=["PatientId", "VisitId"], how="left")
                    )
                    final_batch = (
                        pd.concat([final_batch.loc[~_missing], _filled], ignore_index=True)
                        .sort_values(["PatientId", "VisitId", "StartDateTime"])
                        .reset_index(drop=True)
                    )

            # Write batch parquet — cast Value to str to avoid ArrowInvalid on
            # mixed bool/float/str from specials vs mediator_events columns
            final_batch["Value"] = final_batch["Value"].astype(str)
            final_batch.to_parquet(_TMP / f"batch_{b:05d}.parquet", index=False)
            _done.touch()

            pbar.set_postfix({"rows": len(final_batch), "pid_0": batch_pids[0]})
            pbar.update(1)

    # ── Assemble final CSV ─────────────────────────────────────────────────────
    print("\nAssembling final mediator_input.csv...")
    batch_files = sorted(_TMP.glob("batch_*.parquet"))
    if not batch_files:
        raise RuntimeError("No batch files found — all batches may have been empty.")

    all_batches = pd.concat(
        [pd.read_parquet(p) for p in tqdm(batch_files, desc="assembling")],
        ignore_index=True,
    )
    all_batches.to_csv(output_csv, index=False)
    shutil.rmtree(_TMP, ignore_errors=True)
    print(f"Done. {len(all_batches):,} rows written to {output_csv}")



In [ ]:
# Mapped-events distribution by month-year (diagnostic — reads source parquet, independent of any build step)
# Uses column projection on the measurement parquet: only start_datetime is read, so this is fast (~20-30s).
meas_dt = get_data("measurement", columns=["start_datetime"], as_dask=True)
meas_dt["start_datetime"] = dd.to_datetime(meas_dt["start_datetime"], utc=True, errors="coerce")
meas_dt["month_year"] = meas_dt["start_datetime"].map_partitions(
    lambda s: s.dt.tz_convert(None).dt.to_period("M")
)

event_monthly = (
    meas_dt.groupby("month_year")["start_datetime"]
    .count()
    .compute()
    .sort_index()
)

# Drop months before 2020 (bad timestamps / noise)
event_monthly = event_monthly[
    (event_monthly.index >= pd.Period("2022-01", "M")) &
    (event_monthly.index < pd.Period("2027-01", "M"))
]

event_monthly.plot(kind="bar", figsize=(16, 5), color="steelblue")
plt.title(f"Measurement Events Distribution by Month-Year (N={event_monthly.sum():,} rows)")
plt.xlabel("Month-Year")
plt.ylabel("Number of Measurement Rows")
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.tight_layout()
plt.show()

print(f"\nFirst measurement month : {event_monthly.index.min()}")
print(f"Last  measurement month : {event_monthly.index.max()}")
print(f"Total measurement rows  : {event_monthly.sum():,}")


In [ ]:
# 1. First pass: map events to Parquet
index_specs = load_index_specs("explorations")  # your existing loader

# build_mediator_events_parquet handles synchronous scheduling internally.
# No distributed client needed; close it to free worker memory.
client.close()
build_mediator_events_parquet(
    index_specs=index_specs,
    data_dir=data_dir,
    visits_master_path="visits_master.csv",
    events_parquet_path="mediator_events",
)

# Restart distributed client for finalize_mediator_input (batched, uses persist())
from dask.distributed import Client as _DClient
client = _DClient(processes=True)
print("Client restarted:", client)

# 2. Second pass: propagate + specials + final CSV
lookback = {
    "WEIGHT_MEASURE": 180,   # 6 months
    "HEIGHT_MEASURE": 36500, # 100 years - lifetime
    "BMI_MEASURE":    180,   # 6 months
    "HEMOGLOBIN-A1C_MEASURE": 365,   # 12 months
    "GTT_MEASURE":            365,   # 12 months
    "BASE_GLUCOSE_MEASURE":  365,   # 12 months
    "RDW-CV_MEASURE": 365,   # 12 months

    "ALCOHOL_ABUSE": 365,   # 12 months
    "ANEMIA": 180,   # 6 months
    "BYPASS": 36500, # 100 years - lifetime
    "CANCER_DIAGNOSIS": 1825,   # 5 years of possible forward effect
    "CHRONIC_KIDNEY_CONDITION": 36500, # 100 years - lifetime
    "CHRONIC_LIVER_DISEASE": 36500, # 100 years - lifetime
    "DIABETES_DIAGNOSIS": 36500, # 100 years - lifetime
    "DIALYSYS": 36500, # 100 years - lifetime
    "HEART_DISEASE": 36500, # 100 years - lifetime
    "DEMENTIA": 36500, # 100 years - lifetime, also Alzheimer
    "RESPIRATORY_DISEASE": 365,   # 12 months
    "UNEXPECTED_FALLS": 365,   # 12 months
}

finalize_mediator_input(
    events_parquet_path="mediator_events",
    visits_master_path="visits_master.csv",
    output_csv="mediator_input.csv",
    concept_windows=lookback,
    batch_size_patients=1000,
)

## Create Context Static Data

In [ ]:
# -------------------------------------------------------------------
# Types
# -------------------------------------------------------------------
# (concept_name, kind, window_from_admission, keep_or_remove)
ContextSpec = Tuple[str, str, str, str]


# -------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------
def parse_window(window_str: str) -> timedelta:
    """
    Parse windows like '48h', '14d' into a timedelta.
    Supported units: 'h' (hours) and 'd' (days).
    """
    window_str = str(window_str).strip().lower()
    if window_str.endswith("h"):
        hours = float(window_str[:-1])
        return timedelta(hours=hours)
    elif window_str.endswith("d"):
        days = float(window_str[:-1])
        return timedelta(days=days)
    else:
        raise ValueError(f"Unsupported window format: {window_str}")


def _encode_gender(series: pd.Series) -> pd.Series:
    """
    Map gender to numeric:
        F/female -> 0
        M/male   -> 1
        everything else -> -1
    """
    s = series.astype(str).str.strip().str.lower()
    mapping = {"f": 0, "female": 0, "m": 1, "male": 1}
    coded = s.map(mapping)
    coded = coded.fillna(-1).astype(int)
    return coded


def compute_past_6m_counts(
    visits_all: pd.DataFrame,
    visits_relevant: pd.DataFrame,
    lookback_months: int = 6,
) -> pd.DataFrame:
    """
    For each relevant admission, compute in the previous `lookback_months`
    (using ALL visits, not only relevant):

    - total number of visits
    - number of emergency visits
    - number of inpatient visits

    Returns a DataFrame with columns:
      ['person_id', 'visit_id',
       'past6m_visits', 'past6m_emerg', 'past6m_inpt']
    """
    lookback_td = timedelta(days=30 * lookback_months)

    # Ensure datetime and types
    visits_all = visits_all.copy()
    visits_all["start_datetime"] = pd.to_datetime(
        visits_all["start_datetime"], utc=True, errors="coerce"
    )
    visits_relevant = visits_relevant.copy()
    visits_relevant["start_datetime"] = pd.to_datetime(
        visits_relevant["start_datetime"], utc=True, errors="coerce"
    )

    visits_all["person_id"] = visits_all["person_id"].astype(str)
    visits_all["visit_id"] = visits_all["visit_id"].astype(str)
    visits_relevant["person_id"] = visits_relevant["person_id"].astype(str)
    visits_relevant["visit_id"] = visits_relevant["visit_id"].astype(str)

    # Pre-group all visits by person for lookups
    all_by_pid = {
        pid: g.sort_values("start_datetime")
        for pid, g in visits_all.groupby("person_id")
    }

    rows: List[dict] = []

    for pid, g_rel in visits_relevant.groupby("person_id"):
        g_rel = g_rel.sort_values("start_datetime")
        all_vis = all_by_pid.get(pid)
        if all_vis is None:
            # No history at all (should not really happen)
            for _, r in g_rel.iterrows():
                rows.append(
                    {
                        "person_id": r["person_id"],
                        "visit_id": r["visit_id"],
                        "past6m_visits": 0,
                        "past6m_emerg": 0,
                        "past6m_inpt": 0,
                    }
                )
            continue

        vt = all_vis.get("visit_type")
        if vt is not None:
            vt_str = vt.astype(str).str.lower()
            all_vis = all_vis.assign(_visit_type_lower=vt_str)
        else:
            all_vis = all_vis.assign(_visit_type_lower="")

        for _, r in g_rel.iterrows():
            adm_start = r["start_datetime"]
            if pd.isna(adm_start):
                rows.append(
                    {
                        "person_id": r["person_id"],
                        "visit_id": r["visit_id"],
                        "past6m_visits": 0,
                        "past6m_emerg": 0,
                        "past6m_inpt": 0,
                    }
                )
                continue

            window_start = adm_start - lookback_td
            prev = all_vis[
                (all_vis["start_datetime"] < adm_start)
                & (all_vis["start_datetime"] >= window_start)
            ]

            total = len(prev)
            emerg = int(
                prev["_visit_type_lower"].str.contains("emergency", na=False).sum()
            )
            inpt = int(
                prev["_visit_type_lower"].str.contains("inpatient", na=False).sum()
            )

            rows.append(
                {
                    "person_id": r["person_id"],
                    "visit_id": r["visit_id"],
                    "past6m_visits": total,
                    "past6m_emerg": emerg,
                    "past6m_inpt": inpt,
                }
            )

    return pd.DataFrame(rows)


def apply_context_specs(
    context_df: pd.DataFrame,
    events_df: pd.DataFrame,
    specs,
):
    """
    Apply measurement or drug context specs to build static columns.

    Each spec is: (concept_name, kind, window_str, keep_or_remove)

      concept_name:   mediator ConceptName (e.g. 'BMI_MEASURE')
      kind:           'bool' or 'value'
      window_str:     '48h', '14d', etc. (relative to AdmissionStart)
      keep_or_remove: 'keep' or 'remove'

    Returns:
        updated_context_df, concepts_to_drop_from_events
    """
    concepts_to_drop: Set[str] = set()

    # Keys for joining events to context
    key_visits = ["person_id", "visit_id"]
    key_events = ["PatientId", "VisitId"]

    # Ensure events types
    ev = events_df.copy()
    ev["PatientId"] = ev["PatientId"].astype(str)
    ev["VisitId"] = ev["VisitId"].astype(str)
    ev["StartDateTime"] = pd.to_datetime(ev["StartDateTime"], utc=True, errors="coerce")
    ev["AdmissionStart"] = pd.to_datetime(ev["AdmissionStart"], utc=True, errors="coerce")
    ev["Value_num"] = pd.to_numeric(ev.get("Value"), errors="coerce")

    for concept, kind, window_str, keep_flag in specs:
        concept = str(concept)
        kind = kind.lower().strip()
        keep_flag = keep_flag.lower().strip()

        col_name = concept  # static column name; change if you want a prefix

        if keep_flag == "remove":
            concepts_to_drop.add(concept)

        # Parse window
        window_td = parse_window(window_str)

        # Events for this concept & within window
        sub = ev[ev["ConceptName"] == concept].copy()
        if sub.empty:
            # Column exists but all zero / missing
            if kind == "bool":
                context_df[col_name] = 0
            else:
                context_df[col_name] = -1.0
            continue

        dt_offset = sub["StartDateTime"] - sub["AdmissionStart"]
        mask = (dt_offset >= timedelta(0)) & (dt_offset <= window_td)
        sub = sub[mask].copy()

        if sub.empty:
            if kind == "bool":
                context_df[col_name] = 0
            else:
                context_df[col_name] = -1.0
            continue

        if kind == "bool":
            agg = (
                sub.groupby(key_events)
                .size()
                .reset_index(name=col_name)
            )
            agg[col_name] = 1
        elif kind == "value":
            sub = sub.sort_values("StartDateTime")
            agg = (
                sub.groupby(key_events)["Value_num"]
                .last()
                .reset_index(name=col_name)
            )
        else:
            raise ValueError(f"Unknown context kind: {kind}")

        # Merge into context_df on (person_id, visit_id) <-> (PatientId, VisitId)
        context_df = context_df.merge(
            agg[key_events + [col_name]],
            left_on=key_visits,
            right_on=key_events,
            how="left",
        )
        context_df = context_df.drop(columns=key_events, errors="ignore")

        # Fill NaNs appropriately
        if kind == "bool":
            context_df[col_name] = context_df[col_name].fillna(0).astype(int)
        else:
            context_df[col_name] = context_df[col_name].fillna(-1).astype(float)

    return context_df, concepts_to_drop


def add_cci_like_score(
    context_df: pd.DataFrame,
    *,
    cci_weights: dict = None,
    age_col: str = "age_years",
    include_age_weight: bool = True,
    score_col: str = "CCI_LIKE_SCORE",
    category_col: str = "CCI_LIKE_CATEGORY",
) -> pd.DataFrame:
    """
    Adds a CCI-like comorbidity score per admission to the context_df.

    Assumptions:
    - context_df has one row per (PatientId, VisitId).
    - Chronic condition columns are 0/1 or bool, based on first 48h.
    - 'age' is already in years.

    Parameters
    ----------
    context_df : DataFrame
        Context static table (relevant admissions only).
    cci_weights : dict, optional
        Mapping {column_name -> weight}. If None, uses a reasonable default
        based on your condition_context_specs.
    include_age_weight : bool
        If True, adds 1 point per decade above 50 (i.e., 50?59 => +1, 60?69 => +2, ...).
    score_col : str
        Name of the numeric score column to create.
    category_col : str
        Name of the categorical band column to create.

    Returns
    -------
    DataFrame
        Copy of context_df with two extra columns: score_col, category_col.
    """

    if cci_weights is None:
        # Default mapping from your condition context columns to CCI-like weights
        cci_weights = {
            # ~ Charlson-ish structure, but adapted to what you actually have
            "CANCER_DIAGNOSIS": 2,           # malignancy
            "CHRONIC_KIDNEY_CONDITION": 2,   # renal disease
            "CHRONIC_LIVER_DISEASE": 2,      # liver disease (no severity separation here)
            "HEART_DISEASE": 1,              # CHF / ischemic
            "RESPIRATORY_DISEASE": 1,        # COPD / chronic lung
            "DIABETES_DIAGNOSIS": 1,         # diabetes
            "DEMENTIA": 1,                   # dementia
            # ?Soft? comorbidities ? you can tune these
            "ALCOHOL_ABUSE": 1,
            "ANEMIA": 1,
            "UNEXPECTED_FALLS": 1,
        }

    df = context_df.copy()

    # Base score from comorbidities
    score = np.zeros(len(df), dtype=float)

    for col, w in cci_weights.items():
        if col not in df.columns:
            print(f"[INFO] CCI: column '{col}' not found in context_df; skipping.")
            continue

        # Treat NaN as 0; treat any positive value as "present"
        present = df[col].fillna(0)
        # If these are bools, >0 still works fine
        score += (present > 0).astype(int) * w

    # Optional age component (classic CCI: 1 point per decade over 50)
    if include_age_weight and age_col in df.columns:
        age = pd.to_numeric(df[age_col], errors="coerce").fillna(0)
        age_weight = np.clip((age - 50) // 10, 0, None).astype(int)
        score += age_weight
    elif include_age_weight:
        print(f"[WARN] CCI: age column '{age_col}' not found; age not included in score.")

    df[score_col] = score.astype(int)

    # Categorical bands similar to many CCI uses
    bins = [-1, 0, 1, 2, 4, np.inf]
    labels = ["0", "1", "2", "3-4", "5+"]

    df[category_col] = pd.cut(df[score_col], bins=bins, labels=labels)

    return df

# -------------------------------------------------------------------
# Main: build static per-admission context table
# -------------------------------------------------------------------
def create_context_data(
    visits_master_path: str,
    mediator_input_path: str,
    context_specs,
    lookback_months: int = 6,
    output_static_csv: Optional[str] = None,
    overwrite_mediator: bool = True,
):
    """
    Build the static per-admission context table AND optionally clean
    mediator_input.csv from concepts marked 'remove'.

    Inputs:
        visits_master_path  : visits_master.csv (all admissions)
        mediator_input_path : mediator_input.csv (temporal events)
        context_specs       : iterable of (ConceptName, kind, window, keep/remove)
                              kind in {'bool','value'}
                              window like '48h', '14d'
        lookback_months     : history window for visit counts
        output_static_csv   : where to write static context table (optional)
        overwrite_mediator  : if True, rewrite mediator_input_path after dropping
                              all ConceptNames with keep/remove == 'remove'

    Returns:
        context_df, concepts_dropped
    """

    # -------- Load visits master --------
    visits = pd.read_csv(visits_master_path, low_memory=False)
    visits["person_id"] = visits["person_id"].astype(str)
    visits["visit_id"] = visits["visit_id"].astype(str)

    # Ensure relevant_admission flag exists
    if "relevant_admission" not in visits.columns:
        required = [
            "is_inpatient",
            "broken_visit_id"
            "relevant_department",
            "relevant_duration",
            "death_within_48h",
            "adult_at_admission",
            "diabetic_condition",
        ]
        missing = [c for c in required if c not in visits.columns]
        if missing:
            raise RuntimeError(
                "relevant_admission not in visits and cannot be recomputed; "
                f"missing columns: {missing}"
            )
        visits["relevant_admission"] = (
            visits["is_inpatient"]
            & ~visits["broken_visit_id"]
            & visits["relevant_department"]
            & visits["relevant_duration"]
            & ~visits["death_within_48h"]
            & visits["adult_at_admission"]
            & visits["diabetic_condition"]
        )

    rel_visits = visits[visits["relevant_admission"]].copy()
    print("Relevant admissions for context:", len(rel_visits))

    # -------- Demographics: age + gender_code --------
    rel_visits["start_datetime"] = pd.to_datetime(
        rel_visits["start_datetime"], utc=True, errors="coerce"
    )
    rel_visits["birth_datetime"] = pd.to_datetime(
        rel_visits.get("birth_datetime"), utc=True, errors="coerce"
    )

    age_days = (rel_visits["start_datetime"] - rel_visits["birth_datetime"]).dt.days
    age_years = age_days / 365.25
    age_years = age_years.where((age_years >= 0) & (age_years <= 120), np.nan)
    rel_visits["age_years"] = age_years.fillna(-1).astype(float)

    rel_visits["gender_code"] = _encode_gender(rel_visits.get("gender"))

    # -------- Past 6-month visit counts --------
    history_counts = compute_past_6m_counts(
        visits_all=visits,
        visits_relevant=rel_visits,
        lookback_months=lookback_months,
    )

    # Base context frame
    context_df = rel_visits[
        ["person_id", "visit_id", "gender_code", "age_years"]
    ].copy()
    context_df = context_df.merge(
        history_counts, on=["person_id", "visit_id"], how="left"
    )

    for col in ["past6m_visits", "past6m_emerg", "past6m_inpt"]:
        context_df[col] = context_df[col].fillna(0).astype(int)

    print("Context base rows:", len(context_df))

    # -------- Load only needed columns + concepts for context (avoids OOM) --------
    # Chunked read with column projection + concept filter keeps peak RAM low.
    context_concept_names = set(str(spec[0]) for spec in context_specs)

    id_pairs = context_df[["person_id", "visit_id"]].rename(
        columns={"person_id": "PatientId", "visit_id": "VisitId"}
    )
    id_pairs["PatientId"] = id_pairs["PatientId"].astype(str)
    id_pairs["VisitId"]   = id_pairs["VisitId"].astype(str)

    _ctx_chunks = []
    for _chunk in tqdm(
        pd.read_csv(
            mediator_input_path,
            usecols=["PatientId", "VisitId", "ConceptName",
                     "StartDateTime", "AdmissionStart", "Value"],
            chunksize=500_000,
            low_memory=False,
        ),
        desc="loading context events",
    ):
        _chunk["PatientId"] = _chunk["PatientId"].astype(str)
        _chunk["VisitId"]   = _chunk["VisitId"].astype(str)
        _chunk = _chunk[_chunk["ConceptName"].isin(context_concept_names)]
        _chunk = _chunk.merge(id_pairs, on=["PatientId", "VisitId"], how="inner")
        if not _chunk.empty:
            _ctx_chunks.append(_chunk)

    events_ctx = (
        pd.concat(_ctx_chunks, ignore_index=True)
        if _ctx_chunks
        else pd.DataFrame(columns=[
            "PatientId", "VisitId", "ConceptName",
            "StartDateTime", "AdmissionStart", "Value"
        ])
    )
    del _ctx_chunks
    print(f"Events for context: {len(events_ctx):,} rows ({len(context_concept_names)} concepts)")

    # -------- Apply all context specs in one go --------
    context_df, concepts_to_drop = apply_context_specs(
        context_df, events_ctx, context_specs
    )

    # Make sure numeric columns have no NaNs
    for col in context_df.columns:
        if col in ("person_id", "visit_id"):
            continue
        if pd.api.types.is_numeric_dtype(context_df[col]):
            context_df[col] = context_df[col].fillna(-1)

    # -------- Optionally drop concepts from mediator_input.csv (chunked) --------
    if overwrite_mediator and concepts_to_drop:
        print("Dropping concepts from mediator_input:", sorted(concepts_to_drop))
        import os
        _tmp_path = mediator_input_path + ".tmp"
        _first = True
        _total_orig = 0
        _total_kept = 0
        for _chunk in tqdm(
            pd.read_csv(mediator_input_path, chunksize=500_000, low_memory=False),
            desc="rewriting mediator_input",
        ):
            _total_orig += len(_chunk)
            _chunk = _chunk[~_chunk["ConceptName"].isin(concepts_to_drop)]
            _total_kept += len(_chunk)
            _chunk.to_csv(
                _tmp_path, index=False,
                mode="w" if _first else "a",
                header=_first,
            )
            _first = False
        os.replace(_tmp_path, mediator_input_path)
        print(f"Rewrote mediator_input: {_total_kept:,} rows (was {_total_orig:,})")
    
    # -------- Add CCI-like score --------  
    context_df = add_cci_like_score(context_df)

    # -------- Save static context table --------
    if output_static_csv is not None:
        context_df.to_csv(output_static_csv, index=False)
        print("Static context data saved to:", output_static_csv)

    return context_df, concepts_to_drop


In [ ]:
# ============================================================
# Static Features: Data-Driven Quality Cleaning
# (generic - no domain/causality judgment here, just data hygiene.
#  -1 sentinel values are left as-is; handled later in the causality phase.)
# ============================================================

STATIC_ID_COLS = ['visit_id', 'person_id']  # join key / identifier - never treated as a feature
STATIC_SKIP_OUTLIER_CLIP = ['past6m_visits', 'past6m_emerg', 'past6m_inpt']  # counts - extreme values are real signal, not errors

STATIC_HARDCODED_RANGES = {
    'BMI_MEASURE': (10, 100),        # kg/m^2 - rejects the 900000 error, keeps real morbid obesity
    'WEIGHT_MEASURE': (20, 300),     # kg - rejects the 61700 error, keeps real extreme-weight patients
    'HEIGHT_MEASURE': (100, 220),    # cm - adult range
}

def clip_outliers_iqr(df: pd.DataFrame, cols: List[str], factor: float = 3.0) -> pd.DataFrame:
    """
    Purpose: Cap extreme outliers (data-entry errors) without hardcoding physiological ranges.
    Method: Standard IQR rule, widened (factor=3 instead of the usual 1.5) since clinical
            measurements are naturally skewed and we only want to catch clear errors,
            not trim legitimate extreme-but-real values.
    """
    df = df.copy()
    for col in cols:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr == 0:
            continue
        lo, hi = q1 - factor * iqr, q3 + factor * iqr
        df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan
    return df


def drop_zero_variance(df: pd.DataFrame, id_cols: List[str]) -> pd.DataFrame:
    """Drop columns with a single unique value - carry no signal."""
    const_cols = [c for c in df.columns if c not in id_cols and df[c].nunique(dropna=True) <= 1]
    if const_cols:
        print(f"[drop_zero_variance] Dropping constant columns: {const_cols}")
    return df.drop(columns=const_cols)


def drop_redundant_numeric(df: pd.DataFrame, id_cols: List[str], corr_threshold: float = 0.95) -> pd.DataFrame:
    """
    Purpose: Drop numeric columns that are near-duplicates of another column.
    Method: Greedy pairwise correlation pruning - for any pair above corr_threshold,
            drop the second column encountered (keeps the first, stable given a fixed
            column order).
    """
    df = df.copy()
    numeric_cols = [c for c in df.columns if c not in id_cols and pd.api.types.is_numeric_dtype(df[c])]
    corr = df[numeric_cols].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if (upper[col] > corr_threshold).any()]
    if to_drop:
        print(f"[drop_redundant_numeric] Dropping redundant columns (|corr|>{corr_threshold}): {to_drop}")
    return df.drop(columns=to_drop)


def clip_hardcoded_ranges(df: pd.DataFrame, ranges: dict) -> pd.DataFrame:
    """
    Purpose: Cap columns with a well-established physiological range using domain
             knowledge, instead of IQR - for measurements where a purely statistical
             cutoff would incorrectly reject real extreme-but-valid patients.
    """
    df = df.copy()
    for col, (lo, hi) in ranges.items():
        if col not in df.columns:
            continue
        # -1 sentinel is intentionally left alone here - handled in the causality phase
        mask = (df[col] != MISSING_SENTINEL) & ((df[col] < lo) | (df[col] > hi))
        df.loc[mask, col] = np.nan
    return df


def clean_static_data_quality(df: pd.DataFrame, id_cols: List[str] = STATIC_ID_COLS) -> pd.DataFrame:
    """
    Purpose: Generic, data-driven cleaning of the static features table. Measurements with
             a known physiological range use hardcoded bounds (data-driven IQR is too blunt
             for these - it clips real extreme patients, not just data-entry errors); the
             rest still gets IQR clipping. The -1 missing-value sentinel is intentionally
             left untouched - handled explicitly in the causality phase.
    """
    df = df.copy()

    df = clip_hardcoded_ranges(df, STATIC_HARDCODED_RANGES)

    numeric_cols_all = [c for c in df.columns if c not in id_cols and pd.api.types.is_numeric_dtype(df[c])]
    clip_cols = [
        c for c in numeric_cols_all
        if c not in STATIC_SKIP_OUTLIER_CLIP
        and c not in STATIC_HARDCODED_RANGES
        and df[c].nunique(dropna=True) > 2
    ]
    df = clip_outliers_iqr(df, clip_cols)

    df = drop_zero_variance(df, id_cols)
    df = drop_redundant_numeric(df, id_cols)

    return df


In [ ]:
measurement_context_specs = [
    ("BMI_MEASURE",    "value", "48h", "keep"),
    ("WEIGHT_MEASURE", "value", "48h", "keep"),
    ("HEIGHT_MEASURE", "value", "48h", "remove"),
    ("HEMOGLOBIN-A1C_MEASURE", "value", "48h", "remove"),
    ("GTT_MEASURE", "value", "48h", "remove"),
    ("BASE_GLUCOSE_MEASURE", "value", "48h", "keep"),
    ("RDW-CV_MEASURE", "value", "48h", "remove"),
]

condition_context_specs = [
    # Pure background conditions
    ("ALCOHOL_ABUSE", "bool", "48h", "remove"),
    ("ANEMIA", "bool", "48h", "remove"),
    ("ATHEROSCLEROSIS", "bool", "48h", "remove"),
    ("BYPASS", "bool", "48h", "remove"),
    ("CANCER_DIAGNOSIS", "bool", "48h", "remove"),
    ("CHRONIC_KIDNEY_CONDITION", "bool", "48h", "remove"),
    ("CHRONIC_LIVER_DISEASE", "bool", "48h", "remove"),
    ("DIABETES_DIAGNOSIS", "bool", "48h", "keep"),
    ("DIALYSYS", "bool", "48h", "remove"),
    ("HEART_DISEASE", "bool", "48h", "remove"),
    ("DEMENTIA", "bool", "48h", "remove"),
    ("RESPIRATORY_DISEASE", "bool", "48h", "remove"),
    ("UNEXPECTED_FALLS", "bool", "48h", "remove"),

    # "Demoted" complication concepts, found to be too noisy / mostly during admission - cannot be predicted
    ("ACUTE_RESPIRATORY_DISORDER", "bool", "48h", "remove"),
    ("DIABETIC_COMA", "bool", "48h", "remove"),
    ("HEMIPLEGIA", "bool", "48h", "remove"),
    ("NERVOUS_SYSTEM_DISORDER", "bool", "48h", "remove"),
    ("NEUROVASCULAR_COMPLICATION", "bool", "48h", "remove"),
    ("OTHER_COMPLICATION", "bool", "48h", "remove"),
    ("RETINOPATHY", "bool", "48h", "remove"),
    ("SKIN_ULCER", "bool", "48h", "remove"),
]

drugs_context_specs = [
    ("BASAL_HOME_BITZUA", "bool", "48h", "keep"),
    ("SGLT2_HOME_BITZUA", "bool", "48h", "keep"),
    ("SGLT2_HOSPITAL_BITZUA", "bool", "48h", "keep"),
    ("METFORMIN_HOME_BITZUA", "bool", "48h", "keep"),
    ("METFORMIN_HOSPITAL_BITZUA", "bool", "48h", "keep"),
    ("ANTIDIABETIC_HIGH_HYPO_HOME_BITZUA", "bool", "48h", "keep"),
    ("ANTIDIABETIC_HIGH_HYPO_HOSPITAL_BITZUA", "bool", "48h", "keep"),
    ("ASPIRIN_CONTEXT", "bool", "48h", "remove"),
    ("BP_ACE_CONTEXT", "bool", "48h", "remove"),
    ("BP_ARB_CONTEXT", "bool", "48h", "remove"),
    ("BP_BETA-BLOCKERS_CONTEXT", "bool", "48h", "remove"),
    ("BP_CCB_CONTEXT", "bool", "48h", "remove"),
    ("BP_LOOPS_CONTEXT", "bool", "48h", "remove"),
    ("BP_OTHER_CONTEXT", "bool", "48h", "remove"),
    ("BP_THIAZIDES_CONTEXT", "bool", "48h", "remove"),
    ("MH_ANTIDEPRESSANT_CONTEXT", "bool", "48h", "remove"),
    ("MH_ATYPICAL_CONTEXT", "bool", "48h", "remove"),
    ("MH_MOOD_STABLIZER_CONTEXT", "bool", "48h", "remove"),
    ("MH_TYPICAL_CONTEXT", "bool", "48h", "remove"),
    ("P2Y12_CONTEXT", "bool", "48h", "remove"),
    ("STATINE_CONTEXT", "bool", "48h", "remove"),
]

all_context_specs = (
    measurement_context_specs
    + condition_context_specs
    + drugs_context_specs
)

In [ ]:
context_df, concepts_dropped = create_context_data(
    visits_master_path="visits_master.csv",
    mediator_input_path="mediator_input.csv",
    context_specs=all_context_specs,
    lookback_months=6,
    output_static_csv="context_static_input.csv",
    overwrite_mediator=True,
)

context_df = clean_static_data_quality(context_df)
context_df.to_csv("context_static_input_clean.csv", index=False)
print(f"Rows: {len(context_df)}, Columns: {context_df.shape[1]}")

In [ ]:
print("\n=== Null Counts ===")
print(context_df.isnull().sum())

print("=== Descriptive Stats ===")
print(context_df.describe())

In [ ]:
# Validation
def validate_context_vs_temporal(context_df, processed_df, context_specs):
    print("========== Visit ID-level validation ==========")
    ctx_ids = set(context_df["visit_id"].astype(str))
    processed_ids = set(processed_df["VisitId"].astype(str))

    only_in_ctx = ctx_ids - processed_ids
    only_in_processed = processed_ids - ctx_ids

    if only_in_ctx:
        print(f"? {len(only_in_ctx)}/{len(ctx_ids)} VisitIDs only in context_df")
    else:
        print(f"? All ctx_df VisitIDs are in processed_df (Total: {len(ctx_ids)} records)")

    if only_in_processed:
        print(f"? {len(only_in_processed)}/{len(processed_ids)} VisitIDs only in processed_df")
    else:
        print(f"? All processed_df VisitIDs are in ctx_df (Total: {len(processed_ids)} records)")
    
    print("========== Patient ID-level validation ==========")
    ctx_ids = set(context_df["person_id"].astype(str))
    processed_ids = set(processed_df["PatientId"].astype(str))

    only_in_ctx = ctx_ids - processed_ids
    only_in_processed = processed_ids - ctx_ids

    if only_in_ctx:
        print(f"? {len(only_in_ctx)}/{len(ctx_ids)} PatientIDs only in context_df")
    else:
        print(f"? All ctx_df PatientIDs are in processed_df (Total: {len(ctx_ids)} records)")

    if only_in_processed:
        print(f"? {len(only_in_processed)}/{len(processed_ids)} PatientIDs only in processed_df")
    else:
        print(f"? All processed_df PatientIDs are in ctx_df (Total: {len(processed_ids)} records)")
        
    print("\n========== Concept-level validation ==========")
    # Concepts used in context specs
    from collections import namedtuple

    # If you used a NamedTuple like ContextSpec, this will work directly.
    # If you used plain tuples, adapt indices: name = spec[0], action = spec[3]
    try:
        concepts_keep = {spec.concept_name for spec in context_specs if spec.action == "keep"}
        concepts_remove = {spec.concept_name for spec in context_specs if spec.action == "remove"}
    except AttributeError:
        # Fallback for plain 4-tuples: (concept_name, value_type, window_str, action)
        concepts_keep = {spec[0] for spec in context_specs if spec[3] == "keep"}
        concepts_remove = {spec[0] for spec in context_specs if spec[3] == "remove"}

    all_context_concepts = concepts_keep | concepts_remove

    # Concepts actually present in temporal data
    temporal_concepts = set(processed_df["ConceptName"].astype(str).unique())
    print(f"Total concepts in Temporal Data: {len(temporal_concepts)}. This should match the KB.RawConcept")

    # 1) Concepts with "keep" that disappeared from temporal (suspicious)
    missing_kept = concepts_keep - temporal_concepts

    # 2) Concepts with "remove" that are still in temporal (suspicious)
    remaining_removed = concepts_remove & temporal_concepts

    # 3) All context concepts not present in temporal (regardless of action)
    all_missing = all_context_concepts - temporal_concepts

    print(f"Total context concepts: {len(all_context_concepts)}")
    print(f"Concepts in temporal data: {len(all_context_concepts & temporal_concepts)}")
    print(f"Concepts NOT in temporal data: {len(all_missing)}")

    if missing_kept:
        print("\n? Concepts marked 'keep' but NOT found in temporal data:")
        for c in sorted(missing_kept):
            print("  -", c)
    else:
        print("\n? All 'keep' context concepts appear in temporal data.")

    if remaining_removed:
        print("\n? Concepts marked 'remove' but STILL present in temporal data:")
        for c in sorted(remaining_removed):
            print("  -", c)
    else:
        print("\n? All 'remove' context concepts were successfully dropped from temporal data.")

    print("\nAll context concepts NOT in temporal (for manual review):")
    for c in sorted(all_missing):
        print("  -", c)

In [ ]:
import pandas as pd
import os

# Load small dataframes in full (both are tiny)
context_df    = pd.read_csv("context_static_input.csv", low_memory=False)
visits_master = pd.read_csv("visits_master.csv", low_memory=False)

context_df["visit_id"]     = context_df["visit_id"].astype(str)
context_df["person_id"]    = context_df["person_id"].astype(str)
visits_master["visit_id"]  = visits_master["visit_id"].astype(str)
visits_master["person_id"] = visits_master["person_id"].astype(str)

# Relevant visits from visits_master
relevant_mask      = visits_master["relevant_admission"].fillna(False).astype(bool)
relevant_visit_ids = set(visits_master.loc[relevant_mask, "visit_id"])

# Temporal visit IDs — read only 2 cols from 26.8M-row mediator_input (avoids OOM)
_proc_ids = pd.read_csv(
    "mediator_input.csv",
    usecols=["PatientId", "VisitId"],
    dtype=str,
    low_memory=False,
)
temporal_visit_ids = set(_proc_ids["VisitId"])
del _proc_ids

# Valid set = relevant AND have at least one temporal event
# Visits in context_df but NOT in temporal had zero recorded events → drop them.
valid_visit_ids = relevant_visit_ids & temporal_visit_ids

print(f"Relevant visits in visits_master: {len(relevant_visit_ids):,}")
print(f"Visits with temporal events:      {len(temporal_visit_ids):,}")
print(f"Intersection (valid set):         {len(valid_visit_ids):,}")
print(f"Context-only visits (no events):  {len(set(context_df['visit_id']) - temporal_visit_ids):,}  → dropped")

# ── Align context_df (small, safe in-memory) ──────────────────────────────────
context_df_clean = context_df[context_df["visit_id"].isin(valid_visit_ids)].copy()
print(f"\ncontext_df: {len(context_df):,} → {len(context_df_clean):,} rows")
context_df_clean.to_csv("context_static_input.csv", index=False)
print("Saved: context_static_input.csv")

# ── Align mediator_input.csv (chunked — 26.8M rows can't fit in RAM) ──────────
_tmp = "mediator_input.csv.tmp"
_first, _kept, _total = True, 0, 0
for _chunk in pd.read_csv("mediator_input.csv", chunksize=500_000, low_memory=False):
    _chunk["VisitId"] = _chunk["VisitId"].astype(str)
    _total += len(_chunk)
    _chunk = _chunk[_chunk["VisitId"].isin(valid_visit_ids)]
    _kept += len(_chunk)
    _chunk.to_csv(_tmp, index=False, mode="w" if _first else "a", header=_first)
    _first = False
os.replace(_tmp, "mediator_input.csv")
print(f"mediator_input.csv: {_total:,} → {_kept:,} rows")

# ── Validate using 3-col read (PatientId + VisitId + ConceptName) ─────────────
_proc_val = pd.read_csv(
    "mediator_input.csv",
    usecols=["PatientId", "VisitId", "ConceptName"],
    dtype=str,
    low_memory=False,
)
validate_context_vs_temporal(context_df_clean, _proc_val, all_context_specs)
del _proc_val


In [ ]:
import pandas as pd

# ── Reload small frames ────────────────────────────────────────────────────────
visits_master = pd.read_csv("visits_master.csv", low_memory=False)
context_df    = pd.read_csv("context_static_input.csv", low_memory=False)

visits_master["visit_id"]  = visits_master["visit_id"].astype(str)
visits_master["person_id"] = visits_master["person_id"].astype(str)
context_df["visit_id"]     = context_df["visit_id"].astype(str)
context_df["person_id"]    = context_df["person_id"].astype(str)

visits_master["start_datetime"] = pd.to_datetime(visits_master["start_datetime"], errors="coerce", utc=True)
visits_master["end_datetime"]   = pd.to_datetime(visits_master["end_datetime"],   errors="coerce", utc=True)

# Read only PatientId + VisitId from mediator_input (26.8M rows x all cols = OOM)
_proc = pd.read_csv(
    "mediator_input.csv",
    usecols=["PatientId", "VisitId"],
    dtype=str,
    low_memory=False,
)

# ----------------------------
# 1) Relevant admission set
# ----------------------------
relevant_mask      = visits_master["relevant_admission"].fillna(False).astype(bool)
relevant_vm        = visits_master.loc[relevant_mask].copy()
relevant_visit_ids = set(relevant_vm["visit_id"])
print(f"Relevant admissions in visits_master: {len(relevant_visit_ids):,}")

# ----------------------------
# 2) ID overlap diagnostics
# ----------------------------
temporal_vids = set(_proc["VisitId"])
context_vids  = set(context_df["visit_id"])

print(f"Temporal visits NOT in relevant_admission: {len(temporal_vids - relevant_visit_ids):,}")
print(f"Context visits  NOT in relevant_admission: {len(context_vids  - relevant_visit_ids):,}")
print(f"Temporal visits in relevant_admission:     {len(temporal_vids & relevant_visit_ids):,}")
print(f"Context  visits in relevant_admission:     {len(context_vids  & relevant_visit_ids):,}")

# ----------------------------
# 3) Filter to relevant admissions
# ----------------------------
proc_rel       = _proc[_proc["VisitId"].isin(relevant_visit_ids)]
context_df_rel = context_df[context_df["visit_id"].isin(relevant_visit_ids)].copy()
del _proc

print(f"\nAfter filtering: proc_rel={len(proc_rel):,} rows, context_df_rel={len(context_df_rel):,} rows")

# ----------------------------
# 4) Stats
# ----------------------------
def stats_block(tag, proc_df, ctx_df):
    print(f"\n[{tag}]")
    print(f"Temporal: unique VisitId   = {proc_df['VisitId'].nunique():,}")
    print(f"Temporal: unique PatientId = {proc_df['PatientId'].nunique():,}")
    print(f"Context:  unique visit_id  = {ctx_df['visit_id'].nunique():,}")
    print(f"Context:  unique person_id = {ctx_df['person_id'].nunique():,}")

stats_block("Relevant-admission only", proc_rel, context_df_rel)

# ----------------------------
# 5) 2022+ stats  ← fill in your cutoff date
# ----------------------------
cutoff = pd.Timestamp("2022-01-01", tz="UTC")

relevant_2022_vm        = relevant_vm[relevant_vm["start_datetime"] >= cutoff]
relevant_2022_visit_ids = set(relevant_2022_vm["visit_id"])

proc_rel_2022       = proc_rel[proc_rel["VisitId"].isin(relevant_2022_visit_ids)]
context_df_rel_2022 = context_df_rel[context_df_rel["visit_id"].isin(relevant_2022_visit_ids)].copy()

stats_block(f"Relevant-admission AND start >= {cutoff.date()}", proc_rel_2022, context_df_rel_2022)

print(f"\n2022+ unique visit_ids (visits_master): {len(relevant_2022_visit_ids):,}")
print(f"Unique person_id (visits_master):        {relevant_2022_vm['person_id'].nunique():,}")
print(f"Unique PatientId (temporal):             {proc_rel_2022['PatientId'].nunique():,}")
print(f"Unique person_id (context):              {context_df_rel_2022['person_id'].nunique():,}")


In [ ]:
from itertools import islice
import pandas as pd

# Reload small frames
context_df    = pd.read_csv("context_static_input.csv", low_memory=False, dtype=str)
visits_master = pd.read_csv("visits_master.csv",         low_memory=False, dtype=str)

# Read only ID columns from 26.8M-row mediator_input
_proc = pd.read_csv(
    "mediator_input.csv",
    usecols=["PatientId", "VisitId"],
    dtype=str,
    low_memory=False,
)

# ---------- 1. Build key sets (person_id, visit_id) ----------
relevant_visits = visits_master[visits_master["relevant_admission"].fillna(False).astype(bool)]

relev_pairs = set(zip(relevant_visits["person_id"], relevant_visits["visit_id"]))
proc_pairs  = set(zip(_proc["PatientId"], _proc["VisitId"]))
ctx_pairs   = set(zip(context_df["person_id"], context_df["visit_id"]))
del _proc

print("=== Admission-level counts ===")
print(f"Relevant admissions in visits_master: {len(relev_pairs):,}")
print(f"Unique admissions in mediator_input:  {len(proc_pairs):,}")
print(f"Unique admissions in context_df:      {len(ctx_pairs):,}")

# ---------- 2. mediator_input vs relevant admissions ----------
extra_proc   = proc_pairs - relev_pairs
missing_proc = relev_pairs - proc_pairs

print("\n=== mediator_input vs visits_master (relevant) ===")
print(f"Only in mediator_input (should be 0): {len(extra_proc):,}")
print(f"Relevant admissions missing from mediator_input: {len(missing_proc):,}")
for p in islice(extra_proc, 5):
    print("  PatientId, VisitId =", p)

# ---------- 3. context_df vs relevant admissions ----------
extra_ctx   = ctx_pairs - relev_pairs
missing_ctx = relev_pairs - ctx_pairs

print("\n=== context_df vs visits_master (relevant) ===")
print(f"Only in context_df (should be 0): {len(extra_ctx):,}")
print(f"Relevant admissions missing from context_df: {len(missing_ctx):,}")
for p in islice(extra_ctx, 5):
    print("  person_id, visit_id =", p)

# ---------- 4. Mutual consistency (should both be 0 after alignment) ----------
only_proc = proc_pairs - ctx_pairs
only_ctx  = ctx_pairs  - proc_pairs

print("\n=== mediator_input vs context_df (mutual consistency) ===")
print(f"Only in mediator_input: {len(only_proc):,}")
print(f"Only in context_df:     {len(only_ctx):,}")
for p in islice(only_proc, 5):
    print("  PatientId, VisitId =", p)
for p in islice(only_ctx, 5):
    print("  person_id, visit_id =", p)


In [ ]:
client.restart()   # resets worker state and releases most memory
client.close()
gc.collect()

## Exploring if Anti-Diabetics were Stopped in Admission

In [ ]:
visits_master = pd.read_csv("visits_master.csv", low_memory=False)
context_df = pd.read_csv("context_static_input.csv", low_memory=False)

context_df[['visit_id',
            'METFORMIN_HOME_BITZUA', 'METFORMIN_HOSPITAL_BITZUA', 
            'SGLT2_HOME_BITZUA', 'SGLT2_HOSPITAL_BITZUA', 
            'ANTIDIABETIC_HIGH_HYPO_HOME_BITZUA', 'ANTIDIABETIC_HIGH_HYPO_HOSPITAL_BITZUA']].head()

In [ ]:
df = context_df.merge(
    visits_master[["person_id", "visit_id", "start_datetime", "hospital"]],
    on=["person_id", "visit_id"],
    how="left",
    validate="many_to_one"
)
df["start_datetime"] = pd.to_datetime(df["start_datetime"], utc=True)

df = df[
    (df["start_datetime"] >= "2022-08-01") &
    (df["start_datetime"] < "2026-01-01")
].copy()

df["month"] = df["start_datetime"].dt.to_period("M").dt.to_timestamp()

# Any antidiabetic
df["ANY_HOME"] = df["METFORMIN_HOME_BITZUA"] == 1
df["ANY_HOSPITAL"] = df["METFORMIN_HOSPITAL_BITZUA"] == 1

# High hypo antidiabetic
df["HIGH_HYPO_HOME"] = df["ANTIDIABETIC_HIGH_HYPO_HOME_BITZUA"] == 1
df["HIGH_HYPO_HOSPITAL"] = df["ANTIDIABETIC_HIGH_HYPO_HOSPITAL_BITZUA"] == 1

In [ ]:
before_mask = (
    (df["start_datetime"] >= pd.Timestamp("2022-08-01", tz="UTC"))
    & (df["start_datetime"] < pd.Timestamp("2023-08-01", tz="UTC"))
)
after_mask = (
    (df["start_datetime"] >= pd.Timestamp("2025-01-01", tz="UTC"))
    & (df["start_datetime"] < pd.Timestamp("2026-01-01", tz="UTC"))
)
df["period"] = np.select([before_mask, after_mask], ["Before", "After"], default=None)

In [ ]:
general_table = pd.DataFrame({
    "metric": [
        "High hypo meds continued",
        "Metformin continued"
    ],
    "denominator_n": [
        df["HIGH_HYPO_HOME"].sum(),
        df["ANY_HOME"].sum()
    ],
    "continued_n": [
        (df["HIGH_HYPO_HOME"] & df["HIGH_HYPO_HOSPITAL"]).sum(),
        (df["ANY_HOME"] & df["ANY_HOSPITAL"]).sum()
    ]
})

general_table["continuation_%"] = (
    general_table["continued_n"] /
    general_table["denominator_n"] * 100
).round(1)

general_table

In [ ]:
from scipy.stats import norm

def two_prop_ztest(x1, n1, x2, n2):
    """
    Purpose: Test whether a proportion changed between two independent samples.
    Method: Pooled two-proportion z-test with normal approximation.

    Args:
        x1 (int): Successes in sample 1.
        n1 (int): Total in sample 1.
        x2 (int): Successes in sample 2.
        n2 (int): Total in sample 2.

    Returns:
        float: Two-sided p-value, or np.nan if undefined.
    """
    if n1 == 0 or n2 == 0:
        return np.nan
    p1, p2 = x1 / n1, x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    if se == 0:
        return np.nan
    z = (p1 - p2) / se
    return 2 * (1 - norm.cdf(abs(z)))

def sig_stars(p):
    """
    Purpose: Convert a p-value into conventional significance stars.
    Method: Threshold lookup against standard alpha levels.

    Args:
        p (float): p-value.

    Returns:
        str: "***", "**", "*", "ns", or "" if p is NaN.
    """
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

before = df[df["period"] == "Before"]
after = df[df["period"] == "After"]

metrics = [
    ("High hypo meds continued", "HIGH_HYPO_HOME", "HIGH_HYPO_HOSPITAL"),
    ("Metformin continued", "ANY_HOME", "ANY_HOSPITAL"),
]

print("=========== Continuation before vs after intervention ===========")
for label, home_col, hosp_col in metrics:
    before_denom = before[home_col].sum()
    before_num = (before[home_col] & before[hosp_col]).sum()
    before_pct = before_num / before_denom * 100 if before_denom else np.nan

    after_denom = after[home_col].sum()
    after_num = (after[home_col] & after[hosp_col]).sum()
    after_pct = after_num / after_denom * 100 if after_denom else np.nan

    pval = two_prop_ztest(before_num, before_denom, after_num, after_denom)
    stars = sig_stars(pval)
    p_str = f"p={pval:.3f}" if pd.notna(pval) else "p=n/a"

    print(f"\n{label}:")
    print(f"  Before: {before_num:,}/{before_denom:,} = {before_pct:.1f}%")
    print(f"  After:  {after_num:,}/{after_denom:,} = {after_pct:.1f}%")
    print(f"  {p_str} {stars}")

In [ ]:
hospital_table = (
    df
    .groupby("hospital")
    .apply(lambda g: pd.Series({
        # High hypo
        "high_hypo_home_n": g["HIGH_HYPO_HOME"].sum(),
        "high_hypo_continued_%": (
            (g["HIGH_HYPO_HOME"] & g["HIGH_HYPO_HOSPITAL"]).sum()
            / g["HIGH_HYPO_HOME"].sum() * 100
            if g["HIGH_HYPO_HOME"].sum() > 0 else float("nan")
        ),

        # Any antidiabetic
        "metformin_home_n": g["ANY_HOME"].sum(),
        "metformin_continued_%": (
            (g["ANY_HOME"] & g["ANY_HOSPITAL"]).sum()
            / g["ANY_HOME"].sum() * 100
            if g["ANY_HOME"].sum() > 0 else float("nan")
        )
    }))
    .reset_index()
)

hospital_table[[
    "hospital",
    "high_hypo_home_n",
    "high_hypo_continued_%",
    "metformin_home_n",
    "metformin_continued_%"
]] = hospital_table[[
    "hospital",
    "high_hypo_home_n",
    "high_hypo_continued_%",
    "metformin_home_n",
    "metformin_continued_%"
]].round(1)

hospital_table

In [ ]:
# before_mask, after_mask, and df["period"] are defined earlier, right after df is built
df_period = df[df["period"].isin(["Before", "After"])].copy()

agg = (
    df_period
    .groupby(["hospital", "period"])
    .apply(lambda g: pd.Series({
        "high_hypo_rate": (
            (g["HIGH_HYPO_HOME"] & g["HIGH_HYPO_HOSPITAL"]).sum()
            / g["HIGH_HYPO_HOME"].sum()
            if g["HIGH_HYPO_HOME"].sum() > 0 else np.nan
        ),
        "metformin_rate": (
            (g["ANY_HOME"] & g["ANY_HOSPITAL"]).sum()
            / g["ANY_HOME"].sum()
            if g["ANY_HOME"].sum() > 0 else np.nan
        ),
        "n_high_hypo_home": g["HIGH_HYPO_HOME"].sum(),
        "n_metformin_home": g["ANY_HOME"].sum()
    }))
    .reset_index()
)

agg.loc[agg["n_high_hypo_home"] < 20, "high_hypo_rate"] = np.nan

hospitals = (
    agg["hospital"]
    .dropna()
    .unique()
)[:6]   # ensure 3x2 grid

In [ ]:
from scipy.stats import norm

def two_prop_ztest(x1, n1, x2, n2):
    """
    Purpose: Test whether the continuation rate changed between Before and After.
    Method: Pooled two-proportion z-test with normal approximation.

    Args:
        x1 (int): Successes (continued) in Before sample.
        n1 (int): Total in Before sample.
        x2 (int): Successes (continued) in After sample.
        n2 (int): Total in After sample.

    Returns:
        float: Two-sided p-value, or np.nan if undefined.
    """
    if n1 == 0 or n2 == 0:
        return np.nan
    p1, p2 = x1 / n1, x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    if se == 0:
        return np.nan
    z = (p1 - p2) / se
    return 2 * (1 - norm.cdf(abs(z)))

def sig_stars(p):
    """
    Purpose: Convert a p-value into conventional significance stars.
    Method: Threshold lookup against standard alpha levels.

    Args:
        p (float): p-value.

    Returns:
        str: "***", "**", "*", "ns", or "" if p is NaN.
    """
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

# Anonymized labels for plotting; real names kept in hospital_labels for the printed mapping
hospital_labels = {hosp: f"Hospital{i + 1}" for i, hosp in enumerate(hospitals)}

before_color, after_color = "#4C72B0", "#15BB47"

def plot_continuation(rate_col, n_col, title):
    """
    Purpose: Plot Before vs After continuation rate per hospital, with n and significance.
    Method: 3x2 grid of bar charts, colored by period, with a two-proportion z-test per hospital.

    Args:
        rate_col (str): Column in `agg` holding the continuation rate.
        n_col (str): Column in `agg` holding the home-medication denominator.
        title (str): Figure suptitle.

    Returns:
        None: Displays the figure.
    """
    fig, axes = plt.subplots(3, 2, figsize=(14, 13), sharey=True)
    axes = axes.flatten()

    x = np.arange(2)  # Before, After
    bar_width = 0.6

    for ax, hosp in zip(axes, hospitals):
        sub = agg[agg["hospital"] == hosp]
        before = sub[sub["period"] == "Before"]
        after = sub[sub["period"] == "After"]

        n_before = int(before[n_col].values[0]) if len(before) else 0
        n_after = int(after[n_col].values[0]) if len(after) else 0
        rate_before = before[rate_col].values[0] if len(before) else np.nan
        rate_after = after[rate_col].values[0] if len(after) else np.nan

        # Recover counts from rate * n (rates were computed from exact integer counts)
        x_before = round(rate_before * n_before) if pd.notna(rate_before) else np.nan
        x_after = round(rate_after * n_after) if pd.notna(rate_after) else np.nan

        if pd.notna(x_before) and pd.notna(x_after):
            pval = two_prop_ztest(x_before, n_before, x_after, n_after)
        else:
            pval = np.nan
        stars = sig_stars(pval)
        p_str = f"p={pval:.3f}" if pd.notna(pval) else "p=n/a"

        vals = [
            rate_before * 100 if pd.notna(rate_before) else np.nan,
            rate_after * 100 if pd.notna(rate_after) else np.nan
        ]

        ax.bar(x, vals, bar_width, color=[before_color, after_color])

        ax.set_title(
            f"{hospital_labels[hosp]}\n"
            f"n(Before)={n_before}, n(After)={n_after}\n"
            f"{p_str} {stars}",
            fontsize=10
        )
        ax.set_xticks(x)
        ax.set_xticklabels(["Before", "After"])
        ax.set_ylim(0, 100)
        ax.grid(True, axis="y", alpha=0.3)

    fig.legend(
        handles=[
            plt.Rectangle((0, 0), 1, 1, color=before_color, label="Before"),
            plt.Rectangle((0, 0), 1, 1, color=after_color, label="After"),
        ],
        loc="lower center",
        ncol=2,
        frameon=False
    )
    fig.suptitle(title, fontsize=14)
    plt.tight_layout(rect=[0, 0.04, 1, 0.95])
    plt.show()

plot_continuation(
    "high_hypo_rate", "n_high_hypo_home",
    "High hypo meds continuation by hospital\nBefore vs After July 2023"
)
plot_continuation(
    "metformin_rate", "n_metformin_home",
    "Metformin continuation by hospital\nBefore vs After July 2023"
)

print("Hospital mapping:")
for hosp, label in hospital_labels.items():
    print(f"{label}: {hosp}")

## Exploring Final Cohort

In [ ]:
# Date range parameters -- edit these to change filtering for the whole cell
window_start = pd.Timestamp("2022-08-01", tz="UTC")  # overall analysis window start (inclusive)
window_end   = pd.Timestamp("2026-01-01", tz="UTC")  # overall analysis window end (exclusive)
before_start = pd.Timestamp("2022-08-01", tz="UTC")  # "Before" period start (inclusive)
before_end   = pd.Timestamp("2023-08-01", tz="UTC")  # "Before" period end (exclusive)
after_start  = pd.Timestamp("2025-01-01", tz="UTC")  # "After" period start (inclusive)
after_end    = pd.Timestamp("2026-01-01", tz="UTC")  # "After" period end (exclusive)

# 0) Unfiltered baseline: full visits_master, before any date restriction
vm_unfiltered = visits_master[visits_master['relevant_admission'].fillna(False)].copy()
n_admissions_unfiltered = vm_unfiltered["visit_id"].nunique()
n_patients_unfiltered = vm_unfiltered["person_id"].nunique()
print("=========== Unfiltered (all diabetes visits based on cohort criteria) ===========")
print(f"Admissions: {n_admissions_unfiltered:,}")
print(f"Unique patients: {n_patients_unfiltered:,}")

# 1) Restrict to the analysis window
start_dt_full = pd.to_datetime(vm_unfiltered["start_datetime"], utc=True)
window_mask = (start_dt_full >= window_start) & (start_dt_full < window_end)
vm = vm_unfiltered[window_mask].copy()

# 2) Normalize booleans (handles True/False/NaN safely)
cols = ["has_diabetes_dx", "has_repeated_high_low", "has_extreme_glucose"]
for c in cols:
    vm[c] = vm[c].fillna(False).astype(bool)

n_patients = vm["person_id"].nunique()
n = len(vm)
print(f"\n=========== Relevant admissions (windowed to {window_start.date()} - {window_end.date()}) ===========")
print(f"Relevant admissions rows: {n:,}")
print(f"Unique patients: {n_patients:,}")

# 3) Before vs after intervention, each as its own date range (not a single cut point)
vm_start_dt = pd.to_datetime(vm["start_datetime"], utc=True)
vm_before_mask = (vm_start_dt >= before_start) & (vm_start_dt < before_end)
vm_after_mask = (vm_start_dt >= after_start) & (vm_start_dt < after_end)

before_n_admissions = vm.loc[vm_before_mask, "visit_id"].nunique()
before_n_patients = vm.loc[vm_before_mask, "person_id"].nunique()
after_n_admissions = vm.loc[vm_after_mask, "visit_id"].nunique()
after_n_patients = vm.loc[vm_after_mask, "person_id"].nunique()

print("\n=========== Before vs After intervention ===========")
print(f"Before ({before_start.date()} - {before_end.date()}): {before_n_admissions:,} admissions, {before_n_patients:,} patients")
print(f"After ({after_start.date()} - {after_end.date()}): {after_n_admissions:,} admissions, {after_n_patients:,} patients")

# 4) Relevant admissions & patients per hospital (windowed)
hospital_counts = (
    vm.groupby("hospital")
    .agg(
        n_relevant_admissions=("visit_id", "nunique"),
        n_patients=("person_id", "nunique")
    )
    .reset_index()
    .sort_values("n_relevant_admissions", ascending=False)
)
print("\n=========== Relevant admissions & patients per hospital ===========")
display(hospital_counts)

# 5) Demographics: gender distribution, age, death rate (relevant admissions, windowed)
gender_counts = vm["gender"].value_counts(dropna=False)
gender_pct = vm["gender"].value_counts(dropna=False, normalize=True) * 100

age_years = (
    (pd.to_datetime(vm["start_datetime"], utc=True) - pd.to_datetime(vm["birth_datetime"], utc=True))
    .dt.days / 365.25
)

print("\n=========== Demographics (relevant admissions, windowed) ===========")
print("Gender distribution:")
for g in gender_counts.index:
    print(f"  {g}: {gender_counts[g]:,} ({gender_pct[g]:.2f}%)")
print(
    f"Age (years): mean={age_years.mean():.1f}, median={age_years.median():.1f}, "
    f"min={age_years.min():.1f}, max={age_years.max():.1f}"
)

# Death rate is per patient, not per admission: a patient counts as a death if any of their admissions died within 30d
patient_died = vm.groupby("person_id")["death_within_30d"].any()
print(f"Death rate (death_within_30d): {patient_died.sum():,}/{n_patients:,} patients = {patient_died.mean()*100:.2f}%")

# Helper for ratios
def ratio(num_mask, denom_mask, label):
    """
    Purpose: Print the conditional rate num_mask | denom_mask as a fraction and percentage.
    Method: Count rows satisfying both masks over rows satisfying the denominator mask.

    Args:
        num_mask (pd.Series): Boolean mask for the numerator condition.
        denom_mask (pd.Series): Boolean mask for the denominator (conditioning) condition.
        label (str): Description printed alongside the ratio.

    Returns:
        None: Prints the ratio.
    """
    denom = denom_mask.sum()
    num = (num_mask & denom_mask).sum()
    pct = (num / denom * 100) if denom else 0.0
    print(f"{label}: {num:,}/{denom:,} = {pct:.2f}%")

print("\n=========== Conditional rates (relevant admissions, windowed) ===========")

glucose_mask = vm["has_repeated_high_low"] | vm["has_extreme_glucose"]
dx_mask = vm["has_diabetes_dx"]

# A) From has_repeated_high_low, how many have has_diabetes_dx
ratio(
    num_mask=vm["has_diabetes_dx"],
    denom_mask=vm["has_repeated_high_low"],
    label="P(has_diabetes_dx | has_repeated_high_low)"
)

# B) From has_extreame_glucose, how many have has_diabetes_dx
ratio(
    num_mask=vm["has_diabetes_dx"],
    denom_mask=vm["has_extreme_glucose"],
    label="P(has_diabetes_dx | has_extreme_glucose)"
)

# C) From has_diabetes_dx, how many have (has_repeated_high_low OR has_extreame_glucose)
ratio(
    num_mask=glucose_mask,
    denom_mask=dx_mask,
    label="P(has_repeated_high_low OR has_extreme_glucose | has_diabetes_dx)"
)

# D) From NOT has_diabetes_dx, how many have (has_repeated_high_low OR has_extreame_glucose)
ratio(
    num_mask=glucose_mask,
    denom_mask=~dx_mask,
    label="P(has_repeated_high_low OR has_extreme_glucose | NOT has_diabetes_dx)"
)

# E) From (has_repeated_high_low OR has_extreame_glucose), how many have has_diabetes_dx
ratio(
    num_mask=dx_mask,
    denom_mask=glucose_mask,
    label="P(has_diabetes_dx | has_repeated_high_low OR has_extreme_glucose)"
)

# F) From (has_repeated_high_low OR has_extreame_glucose), how many DON'T have has_diabetes_dx
ratio(
    num_mask=~dx_mask,
    denom_mask=glucose_mask,
    label="P(NOT has_diabetes_dx | has_repeated_high_low OR has_extreme_glucose)"
)

print("\nBase rates in relevant admissions:")
for c in cols:
    print(f"{c}: {vm[c].sum():,}/{n:,} = {vm[c].mean()*100:.2f}%")

# has_diabetes_dx at the patient level (a patient counts if any of their admissions has the flag)
patient_has_dx = vm.groupby("person_id")["has_diabetes_dx"].any()
print(f"has_diabetes_dx (unique patients): {patient_has_dx.sum():,}/{n_patients:,} = {patient_has_dx.mean()*100:.2f}%")

## Clustering Dosages Based on Bitzua


The purpose of this section is to look at the distribution of numeric values within a single ConceptName groups to assess 2 things:

1. Non realistic outlayers, which can mean a wrong code was attributed to a concept, a normalization target was missed or that a data point is contaminated by bad documentation of the medical staff

2. Thresholding objectives - next phase is building the TAK - KBTA knowledge base. Important to validate the expected thresholds are really viewed from the data

### KMEANS

In [ ]:
def cluster_dosage_bins(
    df: pd.DataFrame,
    concept_name: str,
    n_clusters: int = 5,
    log: bool = False,
    cluster_by_id: bool = False,
    verbose: bool = True
):
    """
    One-dimensional k-means clustering on dosage values for the specified 'concept_name'.
    If cluster_by_id=False (default), we combine all dosage values across all db_concept_id.
    If cluster_by_id=True, we do separate clustering per db_concept_id.

    Optionally applies a log transform (log1p).
    Returns a dictionary describing the bin edges (midpoints), cluster centers, 
    and counts. The structure differs slightly based on cluster_by_id:

    - If cluster_by_id=False:
        {
          'global': {
             'edges': [...],
             'centers': [...],
             'counts': [...],
             'labels': <array of cluster indexes in sorted order>,
             'log': bool
          }
        }
    
    - If cluster_by_id=True:
        {
          db_id_1: { 'edges': [...], 'centers': [...], 'counts': [...], 'labels': <array>, 'log': bool },
          db_id_2: { ... },
          ...
        }

    Each set of edges is built by sorting the cluster centers, 
    then creating midpoints. The first/last edge is -inf/+inf by default.
    """

    # Filter the rows with the chosen concept
    df_sub = df[df['ConceptName'] == concept_name].copy()
    if df_sub.empty:
        if verbose:
            print(f"[WARNING] No rows for ConceptName='{concept_name}'. Returning empty.")
        return {}

    # Enforce numeric 'Value'
    df_sub['Value'] = pd.to_numeric(df_sub['Value'], errors='coerce')
    df_sub = df_sub.dropna(subset=['Value'])
    if df_sub.empty:
        if verbose:
            print(f"[WARNING] All 'Value' are NaN for ConceptName='{concept_name}'.")
        return {}

    # We'll define a helper function to do 1D kmeans + edges
    def cluster_id_kmeans(values: np.ndarray) -> dict:
        # Optionally log
        if log:
            data = np.log1p(values.reshape(-1,1))
        else:
            data = values.reshape(-1,1)
        values = values[~np.isnan(values)]
        if len(np.unique(values)) < n_clusters:
            sorted_vals = np.sort(np.unique(values)).tolist()
            edges = [-float('inf')] + [
                (sorted_vals[i] + sorted_vals[i+1]) / 2 for i in range(len(sorted_vals) - 1)
            ] + [float('inf')]
            return {
            'edges': edges,
            'range': [min(values), max(values)],
            'centers': sorted_vals,
            'counts': ["raw"],
            'log': 0
        }
        km = KMeans(n_clusters=n_clusters, random_state=42)
        km.fit(data)
        labels = km.labels_
        centers = km.cluster_centers_.flatten()

        # Undo log if needed
        if log:
            centers = np.expm1(centers)

        # Sort the centers
        sorted_centers = np.sort(centers)
        sorted_centers = np.round(np.sort(centers), 2)

        # Build edges from -inf -> midpoints -> +inf
        edges = [float('-inf')]
        for i in range(len(sorted_centers) - 1):
            midpoint = (sorted_centers[i] + sorted_centers[i+1]) / 2
            edges.append(midpoint)
        edges.append(float('inf'))

        # Re-map cluster labels to sorted order
        center_map = list(zip(km.cluster_centers_.flatten(), range(n_clusters)))
        center_map.sort(key=lambda x: x[0])  # sort by actual center val
        old_to_new = {}
        for new_label, cm in enumerate(center_map):
            old_label = cm[1]
            old_to_new[old_label] = new_label
        new_labels = np.array([old_to_new[l] for l in labels])

        cluster_counts = (
            pd.Series(new_labels)
              .value_counts(sort=False)
              .reindex(range(n_clusters), fill_value=0)
              .to_list()
                )

        return {
            'edges': edges,
            'range': [min(values), max(values)],
            'centers': sorted_centers.tolist(),
            'counts': cluster_counts,
            'log': log
            }

    results = {}
    
    if cluster_by_id:
        records = []
        # 1) Group by db_concept_id, cluster each group
        for db_id, group_df in df_sub.groupby('db_concept_id'):
            vals = group_df['Value'].values
            units = set(group_df['unit'].values)
            info = cluster_id_kmeans(vals)
            results[db_id] = info
            cts_str = ", ".join(f"{c}" for c in info['counts'])

            record = {
                "db_concept_id": int(db_id),
                "centers": info['centers'],
                "range": info['range'],
                "counts": f'[{cts_str}]',
                "units": units
            }
            records.append(record)

        if verbose:
            results_df = pd.DataFrame(records)
            display(results_df[:30])
            display(results_df[30:60])
            display(results_df[60:])
            
    else:
        # Combine all dosage values
        all_vals = df_sub['Value'].values
        info = cluster_1d_kmeans(all_vals)
        results['global'] = info
        if verbose:
            cts_str = ", ".join(f"{c}" for c in info['counts'])
            print(f"Global: centers={info['centers']}, counts=[{cts_str}]")

    return results

In [ ]:
concept = 'KETONES_SERUM_MEASURE'
cluster_dosage_bins(processed_df, concept, 4, False, True, True)
# counts_df = (
#     processed_df[processed_df['ConceptName'] == concept]
#     .groupby(["db_concept_id", "unit"])
#     .size()                # this counts how many rows per group
#     .reset_index(name="count")
#     .sort_values(by="count", ascending=False)
# )
# counts_df.head(30)
values = list(processed_df[processed_df['ConceptName'] == concept]['Value'])
print('Min value in concept=', min(values), ' and max value in concept=', max(values))

### KDE

In [ ]:
def kde_bin_edges(values, bandwidth=0.2, peak_prominence=0.01):
    values = np.array(values).reshape(-1, 1)
    kde = KernelDensity(kernel='gaussian', bandwidth=bandwidth).fit(values)

    x_d = np.linspace(values.min(), values.max(), 1000).reshape(-1, 1)
    log_dens = kde.score_samples(x_d)
    dens = np.exp(log_dens)

    peaks, _ = find_peaks(dens)

    if len(peaks) < 2:
        # Fallback: no meaningful peaks ? single bin
        return [float('-inf'), float('inf')], x_d.flatten(), dens

    valleys, _ = find_peaks(-dens)
    valleys = [v for v in valleys if peaks[0] < v < peaks[-1]]

    bin_edges = [float('-inf')] + list(x_d[valleys].flatten()) + [float('inf')]
    return bin_edges, x_d.flatten(), dens

def cluster_dosage_bins_kde(
    df: pd.DataFrame,
    concept_name: str,
    cluster_by_id: bool = False,
    verbose: bool = True,
    bandwidth: float = 0.2,
    peak_prominence: float = 0.01,
    plot: bool = False,
    xlim: tuple = (0, 200)
):
    """
    KDE-based clustering on dosage values for the specified 'concept_name'.
    Supports per-db_concept_id or global clustering.
    Plots KDE curve + bin edges if plot=True and cluster_by_id=False.
    """
    df_sub = df[df['ConceptName'] == concept_name].copy()
    if df_sub.empty:
        if verbose:
            print(f"[WARNING] No rows for ConceptName='{concept_name}'. Returning empty.")
        return {}

    df_sub['Value'] = pd.to_numeric(df_sub['Value'], errors='coerce')
    df_sub = df_sub.dropna(subset=['Value'])

    if df_sub.empty:
        if verbose:
            print(f"[WARNING] All 'Value' are NaN for ConceptName='{concept_name}'.")
        return {}

    results = {}

    if cluster_by_id:
        for db_id, group_df in df_sub.groupby('db_concept_id'):
            vals = group_df['Value'].values
            if len(vals) < 3:
                continue
            edges, x_d, dens = kde_bin_edges(vals, bandwidth, peak_prominence)
            results[db_id] = {
                'edges': edges,
                'x': x_d,
                'density': dens
            }
            if verbose:
                print(f"db_concept_id={db_id}, bin edges={edges}")
    else:
        all_vals = df_sub['Value'].values
        edges, x_d, dens = kde_bin_edges(all_vals, bandwidth, peak_prominence)
        results['global'] = {
            'edges': edges,
            'x': x_d,
            'density': dens
        }
        if verbose:
            print(f"Global bin edges={edges}")

        if plot:
            import matplotlib.pyplot as plt
            plt.figure(figsize=)
            plt.plot(x_d, dens, label='KDE Density')
            for edge in edges[1:-1]:  # skip -inf and inf
                plt.axvline(edge, color='red', linestyle='--', label='Bin edge' if edge == edges[1] else "")
            plt.title(f"KDE + Bin Edges for '{concept_name}'")
            plt.xlabel("Value")
            plt.ylabel("Density")
            plt.legend()
            plt.grid(True)
            plt.tight_layout()
            plt.xlim(xlim)
            plt.show()

    return results

In [ ]:
processed_df = pd.read_csv('processed.csv', low_memory=False)
set(processed_df['ConceptName'])

In [ ]:
concept = 'CREATINE-KINASE_MEASURE'
cluster_dosage_bins_kde(
    processed_df,
    concept_name=concept,
    cluster_by_id=False,
    verbose=True,
    bandwidth=1.0,        # <- Make sure this is a float
    peak_prominence=0.5,
    plot=True,
    xlim=(0, )
)
counts_df = (
    processed_df[processed_df['ConceptName'] == concept]
    .groupby(["db_concept_id", "unit"])
    .size()                # this counts how many rows per group
    .reset_index(name="count")
    .sort_values(by="count", ascending=False)
)
values = list(processed_df[processed_df['ConceptName'] == concept]['Value'])
print('Min value in concept=', min(values), ' and max value in concept=', max(values), ' for count=', len(values))
counts_df.head(30)

In [ ]:
"""
I added a lot of measure concepts. 
 - Create post process to have medication + route under same row
 - Extract non-temporal features of patients
 - Develop ML + DL
"""

In [ ]:
mediator_input = pd.read_csv('mediator_input.csv', low_memory=False)

In [ ]:
irrelevant_concepts = [
'BASAL_ROUTE',
 'BOLUS_ROUTE',
  'HEIGHT_MEASURE',
]
mediator_input = mediator_input[~mediator_input['ConceptName'].isin(irrelevant_concepts)]
print('Input Size:', len(mediator_input))
mediator_input.to_csv('mediator_input.csv', index=False)

In [ ]:
mediator_input

### Assessing TAK Values

Shows the distribution of all the concepts as a one-shot (helps in defining TRENDS parameters)

In [ ]:
# Load the data
mediator_input = pd.read_csv("mediator_input.csv")

# Convert 'Value' to numeric, coercing errors to NaN
mediator_input["Value_numeric"] = pd.to_numeric(mediator_input["Value"], errors='coerce')

# Filter to only rows where conversion was successful
numeric_df = mediator_input.dropna(subset=["Value_numeric"])

# Group by ConceptName and calculate mean and std
summary = numeric_df.groupby("ConceptName")["Value_numeric"].agg(["mean", "std"]).reset_index()

# Calculate the thresholds based on standard deviation
for factor in [1, 1.5, 2, 3]:
    summary[f"{factor}_lower"] = summary["mean"] - factor * summary["std"]
    summary[f"{factor}_upper"] = summary["mean"] + factor * summary["std"]

In [ ]:
summary

## Targets Exploration

This section examines the **complication concepts** that will later serve as prediction targets.
Two key questions need to be answered before finalising which complications to use:

### Goal 1 — Temporal distribution across the hospitalization
For a complication to be a meaningful *prediction* target it must actually arise *during* the
admission, not only be documented at intake (as is common in MIMIC-IV where diagnoses are
back-filled at discharge).
For each concept we plot the histogram of **hours from admission start** at which every event
occurs.  A healthy target should show events spread throughout the stay, not a spike at hour 0.

### Goal 2 — Ratio of "new" complications (first occurrence after 48 h)
A complication that only ever appears in the first 48 hours is either:
- a pre-existing condition documented at admission, or
- a very early acute event that cannot realistically be *predicted* by a model.

For each admission we find the **first occurrence** of the concept and flag it as *new* if it
appears **strictly after 48 hours** from admission start.
The cell reports the fraction of admissions where the complication is new — if this ratio is
near zero, the concept is likely not a useful dynamic prediction target.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm.auto import tqdm

# ==============================================================================
# PARAMETERS  -- edit these before running
# ==============================================================================
CONCEPT_NAME    = "HYPOGLYCEMIA_EVENT"   # concept to inspect
MEDIATOR_PATH   = "mediator_input.csv"   # path to mediator_input
NEW_THRESHOLD_H = 48                     # hours after which we call it 'new'
BIN_WIDTH_H     = 12                     # histogram bin width in hours
MAX_STAY_H      = 30 * 24               # cap x-axis at 30 days
# ==============================================================================

# -- 1. Load only this concept's rows (chunked -- avoids OOM on 26.8M rows) --
print(f"Loading '{CONCEPT_NAME}' from {MEDIATOR_PATH} ...")
chunks = []
for _chunk in tqdm(
    pd.read_csv(
        MEDIATOR_PATH,
        usecols=["PatientId", "VisitId", "ConceptName", "StartDateTime", "AdmissionStart", "AdmissionEnd"],
        chunksize=2_500_000,
        low_memory=False,
    ),
    desc="scanning",
):
    sub = _chunk[_chunk["ConceptName"] == CONCEPT_NAME]
    if not sub.empty:
        chunks.append(sub)

if not chunks:
    print(f"No events found for '{CONCEPT_NAME}'. Check the name against mediator_input.")
else:

    df = pd.concat(chunks, ignore_index=True)
    del chunks
    print(f"  {len(df):,} events across {df['VisitId'].nunique():,} unique admissions")

    # -- 2. Parse datetimes & compute hours from admission start --
    for col in ["StartDateTime", "AdmissionStart", "AdmissionEnd"]:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

    df["hours_from_admission"] = (
        (df["StartDateTime"] - df["AdmissionStart"]).dt.total_seconds() / 3600
    )
    df["stay_length_h"] = (
        (df["AdmissionEnd"] - df["AdmissionStart"]).dt.total_seconds() / 3600
    )

    valid = df["hours_from_admission"].notna() & (df["hours_from_admission"] >= 0)
    dropped = (~valid).sum()
    if dropped:
        print(f"  Dropped {dropped:,} rows with undefined/negative timing")
    df = df[valid].copy()
    df["hours_clipped"] = df["hours_from_admission"].clip(upper=MAX_STAY_H)

    # -- 3. Per-admission first-occurrence analysis --
    first_occ = (
        df.groupby(["PatientId", "VisitId"])["hours_from_admission"]
        .min()
        .reset_index()
        .rename(columns={"hours_from_admission": "first_hours"})
    )
    n_admissions   = len(first_occ)
    n_new          = (first_occ["first_hours"] > NEW_THRESHOLD_H).sum()
    n_at_admission = (first_occ["first_hours"] <= NEW_THRESHOLD_H).sum()
    pct_new        = 100 * n_new / n_admissions if n_admissions > 0 else 0

    # -- 4. Summary printout --
    sep = "=" * 60
    print(f"\n{sep}")
    print(f"  Concept : {CONCEPT_NAME}")
    print(sep)
    print(f"  Total events                    : {len(df):,}")
    print(f"  Admissions with this concept    : {n_admissions:,}")
    print(f"  First occ <= {NEW_THRESHOLD_H}h (at admission) : {n_at_admission:,}  ({100 - pct_new:.1f}%)")
    print(f"  First occ >  {NEW_THRESHOLD_H}h (new compl.)   : {n_new:,}  ({pct_new:.1f}%)")
    n_events_after = (df["hours_from_admission"] > NEW_THRESHOLD_H).sum()
    pct_events_after = 100 * n_events_after / len(df) if len(df) > 0 else 0
    print(f"  Events after {NEW_THRESHOLD_H}h (all occurrences): {n_events_after:,}  ({pct_events_after:.1f}%)")
    print(f"  Median hours from admission     : {df['hours_from_admission'].median():.1f} h")
    print(f"  Median stay length              : {df['stay_length_h'].median():.1f} h")
    if pct_new < 20:
        print(f"\n  [!] LOW new-ratio ({pct_new:.1f}%) -- likely an admission diagnosis, not a dynamic target.")
    else:
        print(f"\n  [OK] new-ratio {pct_new:.1f}% -- distributes meaningfully across stay.")

    # -- 5. Plots --
    bins = np.arange(0, MAX_STAY_H + BIN_WIDTH_H, BIN_WIDTH_H)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    # Extra top margin so suptitle doesn't sit on the subplot titles
    fig.suptitle(f"Concept: {CONCEPT_NAME}", fontsize=14, fontweight="bold", y=1.04)

    # Left: all events
    ax = axes[0]
    ax.hist(df["hours_clipped"], bins=bins, color="#4c78a8", edgecolor="none", alpha=0.85)
    ax.axvline(NEW_THRESHOLD_H, color="crimson", linewidth=1.5, linestyle="--",
               label=f"{NEW_THRESHOLD_H}h threshold")
    ax.set_xlabel("Hours from admission start")
    ax.set_ylabel("Number of events")
    ax.set_title("Event timing — all occurrences", pad=10)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x)}h"))
    ax.legend()

    # Right: first occurrence per admission
    ax = axes[1]
    first_clipped = first_occ["first_hours"].clip(upper=MAX_STAY_H)
    ax.hist(first_clipped[first_clipped <= NEW_THRESHOLD_H], bins=bins,
            color="#e45756", alpha=0.8, label=f"<={NEW_THRESHOLD_H}h (at admission)")
    ax.hist(first_clipped[first_clipped > NEW_THRESHOLD_H], bins=bins,
            color="#54a24b", alpha=0.8, label=f">{NEW_THRESHOLD_H}h (new)")
    ax.axvline(NEW_THRESHOLD_H, color="black", linewidth=1.5, linestyle="--")
    ax.set_xlabel("Hours from admission start")
    ax.set_ylabel("Number of admissions")
    ax.set_title(f"First occurrence per admission  [{pct_new:.1f}% new]", pad=10)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x)}h"))
    ax.legend()

    plt.tight_layout()
    plt.show()
